# DAgger: train on the boards the policy actually breaks

Run-05 (medium branch-state SFT) is the control case for this notebook:

| Measurement | Result |
| --- | --- |
| Dev loss, 32 unseen puzzles | 0.116 → **0.049** (−58%) |
| Per-token confidence on the correct completion | 89% → 95% |
| Branch decisions changed | **3 of 10 puzzles, one decision each** |
| Branch decisions still correct | 96/173 → 95/163 |
| Deduction actions | 3,753 → 3,975 (+5.9%) |
| Probe solved | **6/10, exactly run-04** |

The labels were right and the optimizer worked, so the epoch did learn — it just
learned the wrong states. Every run-05 training board is one the **oracle** reaches.
Baseline dev loss says the model already put the correct choice first on ~every one
of them (~0.8 nats per 7-token completion), so the only thing left to fit was the
margin. The boards it fails on are the ones it **creates itself** after a wrong
branch, and those never appeared in training.

This notebook records those boards:

1. Roll run-04 out on **32 training puzzles** (plus 8 held out) at depth 2 / 256 actions.
2. Keep every branch state the policy actually reaches, with the audited value as
   the label — so a wrong guess becomes a corrected positive.
3. Train **branch rows only** (`branch_repeat` defaults to 1; both menu orders are
   still generated), 3 epochs at 2e-4.
4. Re-run the same 10-id probe, and compare branch **sequences**, not just counts.

**Expected wall time** on T4 x2: data ~1 min, collection **~70-85 min** (40
puzzles, 2 shards), smoke ~6 min, training **~25-40 min** (566 rows, ~53 steps),
probe ~40 min. Stop after the probe unless it improves on 6/10.

**Inputs** (same two datasets as run-05):

* `run-04-branch-search-depth` — the 7 adapter files, flat;
* your data dataset containing `train-puzzles-medium.jsonl`.

**Success bar.** 7+/10 on the probe with branch sequences that now *differ* from
run-04. 6/10 with different sequences means the policy moved but not enough;
6/10 with identical sequences means the states still were not the binding ones.
Fewer than 6 means the fine-tune damaged a solved puzzle — revert.


## 1. Environment

Two `Tesla T4` (capability `(7, 5)`) -> fp16, the same pins as run-04 and run-05 so
the adapter loads and the chat template renders identically.


In [ ]:
import subprocess, sys, torch

print(sys.version)
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("gpus", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))
if torch.cuda.is_available():
    major = torch.cuda.get_device_capability(0)[0]
    print("selected dtype:", "bfloat16" if major >= 8 else "float16 (T4 is sm_75)")
print()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


## 2. Dependencies

Same pins as run-05. Nothing new is needed: the collector is plain Python plus the
adapter the evaluator already loads.


In [ ]:
%pip install -q "transformers==5.22.2" "trl>=1.15.0,<2" "peft>=0.21.2,<1" "datasets>=4.8.4,<5" "accelerate>=1.15.0,<2" "huggingface-hub>=1.31.0,<2" "safetensors>=0.8.0,<1"
%pip uninstall -y -q torchao


In [ ]:
import importlib.metadata as metadata

for package in ("torch", "transformers", "trl", "peft", "accelerate", "datasets"):
    print(f"{package:14s}", metadata.version(package))


## 3. Locate the adapter and the medium data

Discovery is by filename, so it works whether Kaggle mounts a dataset at
`/kaggle/input/run-04-branch-search-depth` or under
`/kaggle/input/datasets/<user>/...`. If it picks the wrong copy, set
`ADAPTER_OVERRIDE` or `DATA_OVERRIDE` and re-run.


In [ ]:
from pathlib import Path
import json

INPUT = Path("/kaggle/input")
ADAPTER_OVERRIDE = None   # e.g. Path("/kaggle/input/run-04-branch-search-depth")
DATA_OVERRIDE = None      # e.g. Path("/kaggle/input/train-tracks-data")

adapter_hits = sorted({hit.parent for hit in INPUT.glob("**/adapter_config.json")})
print("adapter_config.json found in:")
for hit in adapter_hits:
    marker = (hit / "adapter_model.safetensors").exists()
    print("  ", hit, "| weights:", marker)
assert adapter_hits, (
    "No adapter_config.json under /kaggle/input. Upload the 7 run-04 adapter files "
    "as a dataset (not the zip), and attach it.")
preferred = [hit for hit in adapter_hits if "run-04" in hit.name.lower()]
ADAPTER = (Path(ADAPTER_OVERRIDE) if ADAPTER_OVERRIDE
           else (preferred[0] if preferred else adapter_hits[0]))
assert (ADAPTER / "adapter_model.safetensors").exists(), (
    f"{ADAPTER} has a config but no adapter_model.safetensors")

config = json.loads((ADAPTER / "adapter_config.json").read_text())
print("\nadapter:", ADAPTER)
print("  base model :", config.get("base_model_name_or_path"))
print("  r / alpha  :", config.get("r"), "/", config.get("lora_alpha"))
print("  targets    :", len(config.get("target_modules") or []), "modules")
print("  tokenizer  :", (ADAPTER / "tokenizer.json").exists())
assert config.get("r") == 16, "expected the run-04 adapter (r=16)"

data_hits = sorted({hit.parent for hit in INPUT.glob("**/train-puzzles-medium.jsonl")})
print("\ntrain-puzzles-medium.jsonl found in:")
for hit in data_hits:
    print("  ", hit)
assert data_hits, (
    "No train-puzzles-medium.jsonl under /kaggle/input. Attach the data dataset, "
    "or set DATA_OVERRIDE.")
DATA = Path(DATA_OVERRIDE) if DATA_OVERRIDE else data_hits[0]
print("\nusing DATA =", DATA)

MODEL = config.get("base_model_name_or_path") or "Qwen/Qwen2.5-1.5B-Instruct"
print("base model  :", MODEL)


## 4. Write the code into `/kaggle/working/tt`

Six modules. `tt_dagger.py` is new: it wraps any policy, records the board at every
branch decision, and refuses to use a state whose audited value is no longer in the
menu (a poisoned board cannot be taught the right answer).


In [ ]:
from pathlib import Path

TT = Path("/kaggle/working/tt")
TT.mkdir(parents=True, exist_ok=True)
print(TT)


| File | Contents |
| --- | --- |
| `tt_core.py` | rule engine, board helpers, action menu / prompt / validator |
| `tt_lib.py` | branch search + branch-focused data builder |
| `tt_prep.py` | oracle traces, three-way split, JSONL writer |
| `tt_dagger.py` | policy-state collector + branch-only example builder |
| `tt_train.py` | LoRA SFT CLI (`--examples`, `--init-adapter`) |
| `tt_eval.py` | rollout evaluator with the optional branch filter |


In [ ]:
%%writefile /kaggle/working/tt/tt_core.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Small, inspectable constraint engine for offline Train Tracks traces.

Rows and columns are zero-based internally. Masks use the tile order in TILES.
Every deduction removes only candidates ruled out by a visible local constraint.
"""

from __future__ import annotations

from dataclasses import dataclass

TILES = ("XX", "NS", "EW", "NE", "NW", "SE", "SW")
BIT = {name: 1 << index for index, name in enumerate(TILES)}
ALL = (1 << len(TILES)) - 1
TRACK = ALL ^ BIT["XX"]
DIRECTIONS = {"N": (-1, 0, "S"), "E": (0, 1, "W"),
              "S": (1, 0, "N"), "W": (0, -1, "E")}


def choices(mask: int) -> list[str]:
    return [name for name in TILES if mask & BIT[name]]


def one(mask: int) -> str | None:
    return TILES[mask.bit_length() - 1] if mask and mask.bit_count() == 1 else None


def clue_from_record(record: dict) -> dict:
    puzzle = record["puzzle"]
    height = len(puzzle["rows"])
    givens = [(height - 1 - row, column, shape)
              for shape, row, column, *_ in puzzle["pieces"]]
    return {"rows": puzzle["rows"], "columns": puzzle["columns"],
            "given_shapes": sorted(givens)}


def initial_domains(clue: dict) -> list[list[int]]:
    height, width = len(clue["rows"]), len(clue["columns"])
    given = {(r, c): shape for r, c, shape in clue["given_shapes"]}
    board = []
    for r in range(height):
        line = []
        for c in range(width):
            if (r, c) in given:
                line.append(BIT[given[r, c]])
                continue
            allowed = 0
            for name in TILES:
                if all(0 <= r + DIRECTIONS[d][0] < height
                       and 0 <= c + DIRECTIONS[d][1] < width for d in name if d != "X"):
                    allowed |= BIT[name]
            line.append(allowed)
        board.append(line)
    return board


def snapshot(board: list[list[int]]) -> list[list[int]]:
    return [line.copy() for line in board]


@dataclass(frozen=True)
class Step:
    kind: str  # deduce or contradiction
    rule: str
    anchor: dict
    cell: tuple[int, int] | None = None
    mask: int | None = None

    def action(self) -> dict:
        result = {"type": self.kind, "rule": self.rule, "anchor": self.anchor}
        if self.cell is not None:
            result["cell"] = {"row": self.cell[0] + 1,
                              "column": self.cell[1] + 1}
        if self.mask is not None:
            result["candidates"] = choices(self.mask)
        return result


def _line_step(board, cells, clue, kind, index):
    fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
    possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
    anchor = {kind: index + 1, "clue": clue}
    if fixed > clue or possible < clue:
        return Step("contradiction", "clue_bounds", anchor)
    for r, c in cells:
        old = board[r][c]
        if fixed == clue and old & BIT["XX"] and old & TRACK:
            return Step("deduce", "clue_satisfied", anchor,
                        (r, c), BIT["XX"])
        if possible == clue and old & BIT["XX"] and old & TRACK:
            return Step("deduce", "all_remaining_cells_required", anchor,
                        (r, c), old & TRACK)
    return None


def _port_step(board, r, c):
    height, width = len(board), len(board[0])
    old = board[r][c]
    supported = 0
    for shape in choices(old):
        if shape == "XX":
            ports = ""
        else:
            ports = shape
        valid = True
        for direction, (dr, dc, opposite) in DIRECTIONS.items():
            nr, nc = r + dr, c + dc
            if not (0 <= nr < height and 0 <= nc < width):
                continue  # The boundary and endpoint rule is set initially.
            if not any((direction in ports) == (opposite in neighbor)
                       for neighbor in choices(board[nr][nc])):
                valid = False
                break
        if valid:
            supported |= BIT[shape]
    if supported == old:
        return None
    anchor = {"row": r + 1, "column": c + 1}
    return (Step("deduce", "port_reciprocity", anchor, (r, c), supported)
            if supported else Step("contradiction", "port_reciprocity", anchor,
                                   (r, c)))


def next_step(board: list[list[int]], clue: dict) -> Step | None:
    height, width = len(board), len(board[0])
    for r in range(height):
        event = _line_step(board, [(r, c) for c in range(width)],
                           clue["rows"][r], "row", r)
        if event:
            return event
    for c in range(width):
        event = _line_step(board, [(r, c) for r in range(height)],
                           clue["columns"][c], "column", c)
        if event:
            return event
    for r in range(height):
        for c in range(width):
            event = _port_step(board, r, c)
            if event:
                return event
    if all(one(mask) is not None for line in board for mask in line):
        error = completed_error(board, clue)
        if error:
            return Step("contradiction", error, {"board": "complete"})
    return None


def completed_error(board: list[list[int]], clue: dict) -> str | None:
    height, width = len(board), len(board[0])
    grid = [[one(mask) for mask in line] for line in board]
    if any(tile is None for line in grid for tile in line):
        return "incomplete"
    if any(sum(tile != "XX" for tile in grid[r]) != clue["rows"][r]
           for r in range(height)) or any(
               sum(grid[r][c] != "XX" for r in range(height)) != clue["columns"][c]
               for c in range(width)):
        return "clue_mismatch"
    occupied = {(r, c) for r in range(height) for c in range(width)
                if grid[r][c] != "XX"}
    given = {(r, c): shape for r, c, shape in clue["given_shapes"]}
    if any(grid[r][c] != shape for (r, c), shape in given.items()):
        return "changed_given"
    external = []
    for r, c in occupied:
        for direction in grid[r][c]:
            dr, dc, opposite = DIRECTIONS[direction]
            nr, nc = r + dr, c + dc
            if not (0 <= nr < height and 0 <= nc < width):
                external.append((r, c, direction))
            elif opposite not in grid[nr][nc]:
                return "port_reciprocity"
    expected_external = {(r, c, direction) for (r, c), shape in given.items()
                         for direction in shape if not (
                             0 <= r + DIRECTIONS[direction][0] < height and
                             0 <= c + DIRECTIONS[direction][1] < width)}
    if set(external) != expected_external or len(external) != 2:
        return "endpoints"
    if not occupied:
        return "empty_path"
    found, todo = set(), [next(iter(occupied))]
    while todo:
        cell = todo.pop()
        if cell in found:
            continue
        found.add(cell)
        r, c = cell
        for direction in grid[r][c]:
            dr, dc, _ = DIRECTIONS[direction]
            neighbor = r + dr, c + dc
            if neighbor in occupied and neighbor not in found:
                todo.append(neighbor)
    return None if found == occupied else "disconnected_path"


def apply_deduction(board, step: Step) -> None:
    if step.kind != "deduce" or step.cell is None or step.mask is None:
        raise ValueError("Expected a deduction")
    r, c = step.cell
    if step.mask == 0 or step.mask == board[r][c] or step.mask & board[r][c] != step.mask:
        raise ValueError("Deduction must remove at least one candidate")
    board[r][c] = step.mask


def branch_cell(board: list[list[int]]) -> tuple[int, int] | None:
    candidates = [(mask.bit_count(), r, c) for r, line in enumerate(board)
                  for c, mask in enumerate(line) if mask.bit_count() > 1]
    return min(candidates)[1:] if candidates else None


"""Shared text policy format and rule checker for interactive trace rollouts."""

import hashlib
import json

SYSTEM = ("Solve Train Tracks one action at a time. Choose only from the "
          "provided legal-action menu. Return exactly one JSON object with the "
          "selected id, for example {\"choice\":\"a00\"}. Do not invent an "
          "action or alter a listed action.")


def development_ids(puzzle_ids, count=32):
    ids = sorted(set(puzzle_ids), key=lambda x: hashlib.sha256(x.encode()).hexdigest())
    if len(ids) <= count:
        raise ValueError("Not enough puzzles for a disjoint train/development split")
    return set(ids[:count])


def _action_key(action):
    return json.dumps(action, sort_keys=True, separators=(",", ":"))


def legal_actions(state, pending=False):
    """Enumerate rule-supported actions; offer branches only when propagation stalls."""
    board = state["domains"]
    clue = {"rows": state["row_clues"], "columns": state["column_clues"],
            "given_shapes": [(g["row"] - 1, g["column"] - 1, g["value"])
                             for g in state["givens"]]}
    if pending:
        stack = state["branch_stack"]
        return ([{"type": "undo", "to_step": stack[-1]["to_step"]}]
                if stack else [])

    height, width = len(board), len(board[0])
    stack = state["branch_stack"]
    actions = []
    for row in range(height):
        cells = [(row, c) for c in range(width)]
        fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
        possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
        number = clue["rows"][row]
        anchor = {"row": row + 1, "clue": number}
        if stack and (fixed > number or possible < number):
            actions.append(Step("contradiction", "clue_bounds", anchor).action())
        for c in range(width):
            cell = (row, c)
            for rule in ("clue_satisfied", "all_remaining_cells_required"):
                step = _rule_step(board, clue, rule, anchor, cell)
                if step:
                    actions.append(step.action())
    for column in range(width):
        cells = [(r, column) for r in range(height)]
        fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
        possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
        number = clue["columns"][column]
        anchor = {"column": column + 1, "clue": number}
        if stack and (fixed > number or possible < number):
            actions.append(Step("contradiction", "clue_bounds", anchor).action())
        for r in range(height):
            cell = (r, column)
            for rule in ("clue_satisfied", "all_remaining_cells_required"):
                step = _rule_step(board, clue, rule, anchor, cell)
                if step:
                    actions.append(step.action())
    for r in range(height):
        for c in range(width):
            step = _port_step(board, r, c)
            if step and (step.kind == "deduce" or stack):
                actions.append(step.action())

    error = completed_error(board, clue)
    if error is None:
        actions.append({"type": "finish"})
    elif stack and error != "incomplete":
        actions.append(Step("contradiction", error, {"board": "complete"}).action())

    # A branch is a fallback after no visible deduction remains.
    if not actions:
        cell = branch_cell(board)
        if cell:
            r, c = cell
            for value in choices(board[r][c]):
                actions.append({"type": "branch",
                                "cell": {"row": r + 1, "column": c + 1},
                                "value": value})

    unique = {_action_key(action): action for action in actions}
    return [unique[key] for key in sorted(unique)]


def choice_id(action, actions):
    key = _action_key(action)
    for index, candidate in enumerate(actions):
        if _action_key(candidate) == key:
            return f"a{index:03d}"
    raise ValueError(f"Trace action is absent from its legal menu: {action}")


def prompt(state, actions):
    readable = dict(state)
    readable["domains"] = [[choices(mask) for mask in row]
                           for row in state["domains"]]
    readable["legal_actions"] = [
        {"id": f"a{index:03d}", "action": action}
        for index, action in enumerate(actions)]
    return ("Board state (each domains cell lists its remaining tiles; "
            "XX means empty):\n"
            + json.dumps(readable, separators=(",", ":"), ensure_ascii=False)
            + "\nChoose one listed legal action. JSON choice id only.")


def completion(action_id):
    return json.dumps({"choice": action_id}, separators=(",", ":"))


def parse_action(raw, actions):
    try:
        result = json.loads(raw.strip())
    except (ValueError, TypeError):
        return None
    if not isinstance(result, dict) or set(result) != {"choice"}:
        return None
    choice = result["choice"]
    if not isinstance(choice, str) or not choice.startswith("a"):
        return None
    try:
        index = int(choice[1:])
    except ValueError:
        return None
    return actions[index] if 0 <= index < len(actions) and choice == f"a{index:03d}" else None


def _cell(action, board):
    cell = action.get("cell")
    if not isinstance(cell, dict):
        return None
    r, c = cell.get("row"), cell.get("column")
    if type(r) is not int or type(c) is not int or not (1 <= r <= len(board)) or not (1 <= c <= len(board[0])):
        return None
    return r - 1, c - 1


def _rule_step(board, clue, rule, anchor, cell=None):
    if not isinstance(rule, str):
        return None
    h, w = len(board), len(board[0])
    if rule in {"clue_satisfied", "all_remaining_cells_required", "clue_bounds"}:
        if not isinstance(anchor, dict):
            return None
        if "row" in anchor and type(anchor["row"]) is int and 1 <= anchor["row"] <= h:
            i = anchor["row"] - 1
            cells = [(i, c) for c in range(w)]
            kind, number = "row", clue["rows"][i]
        elif "column" in anchor and type(anchor["column"]) is int and 1 <= anchor["column"] <= w:
            i = anchor["column"] - 1
            cells = [(r, i) for r in range(h)]
            kind, number = "column", clue["columns"][i]
        else:
            return None
        if anchor != {kind: i + 1, "clue": number}:
            return None
        fixed = sum(not (board[r][c] & BIT["XX"]) for r, c in cells)
        possible = sum(bool(board[r][c] & TRACK) for r, c in cells)
        if rule == "clue_bounds":
            return Step("contradiction", rule, anchor) if fixed > number or possible < number else None
        if cell not in cells:
            return None
        r, c = cell
        old = board[r][c]
        if not (old & BIT["XX"] and old & TRACK):
            return None
        if rule == "clue_satisfied" and fixed == number:
            return Step("deduce", rule, anchor, cell, BIT["XX"])
        if rule == "all_remaining_cells_required" and possible == number:
            return Step("deduce", rule, anchor, cell, old & TRACK)
    elif rule == "port_reciprocity" and isinstance(anchor, dict):
        if type(anchor.get("row")) is not int or type(anchor.get("column")) is not int:
            return None
        r, c = anchor["row"] - 1, anchor["column"] - 1
        if not (0 <= r < h and 0 <= c < w):
            return None
        if cell is not None and cell != (r, c):
            return None
        return _port_step(board, r, c)
    elif rule in {"disconnected_path", "endpoints", "empty_path", "clue_mismatch", "changed_given"}:
        if anchor == {"board": "complete"} and completed_error(board, clue) == rule:
            return Step("contradiction", rule, anchor)
    return None


def validate_action(board, clue, action, stack, pending):
    """Return (valid, reason, canonical action). No solution is consulted."""
    if not isinstance(action, dict):
        return False, "malformed", None
    kind = action.get("type")
    if not isinstance(kind, str):
        return False, "malformed", None
    if pending:
        if kind != "undo" or not stack or action.get("to_step") != stack[-1]["to_step"]:
            return False, "expected_undo", None
        return True, "ok", {"type": "undo", "to_step": stack[-1]["to_step"]}
    if kind in {"deduce", "contradiction"}:
        cell = _cell(action, board) if "cell" in action else None
        if "cell" in action and cell is None:
            return False, "invalid_cell", None
        step = _rule_step(board, clue, action.get("rule"), action.get("anchor"), cell)
        if step is None or step.kind != kind:
            return False, "unsupported_rule", None
        if kind == "deduce":
            if cell is None or step.cell != cell or action.get("candidates") != choices(step.mask):
                return False, "wrong_candidates", None
        elif step.cell is not None and step.cell != cell:
            return False, "wrong_contradiction_cell", None
        elif step.cell is None and cell is not None:
            return False, "unexpected_cell", None
        return True, "ok", step.action()
    if kind == "branch":
        cell = _cell(action, board)
        value = action.get("value")
        if cell is None or not isinstance(value, str) or value not in BIT or board[cell[0]][cell[1]].bit_count() < 2:
            return False, "invalid_branch", None
        if not board[cell[0]][cell[1]] & BIT[value]:
            return False, "branch_outside_domain", None
        if completed_error(board, clue) not in {"incomplete", None}:
            return False, "contradictory_board", None
        return True, "ok", {"type": "branch", "cell": action["cell"], "value": value}
    if kind == "finish":
        return (True, "ok", {"type": "finish"}) if completed_error(board, clue) is None else (False, "unfinished", None)
    return False, "unexpected_action", None


def split_examples(rows, max_deductions_per_puzzle=48):
    """Keep every start and rare positive action; balance sampled deductions by rule."""
    from collections import defaultdict
    grouped = defaultdict(list)
    for row in rows:
        if row.get("expert_policy_label"):
            grouped[row["puzzle_id"]].append(row)
    selected = []
    for puzzle_id, items in grouped.items():
        deductions = [r for r in items if r["action"]["type"] == "deduce"]
        rare = [r for r in items if r["action"]["type"] != "deduce"]
        starts = [r for r in items if r["step_index"] == 0]
        selected.extend(starts)

        start_steps = {r["step_index"] for r in starts}
        by_rule = defaultdict(list)
        for row in deductions:
            if row["step_index"] not in start_steps:
                by_rule[row["action"].get("rule", "unknown")].append(row)
        for rule, rule_rows in by_rule.items():
            rule_rows.sort(key=lambda r: hashlib.sha256(
                f"{puzzle_id}:{r['step_index']}:42".encode()).hexdigest())

        start_deductions = sum(r["action"]["type"] == "deduce" for r in starts)
        remaining = max(0, max_deductions_per_puzzle - start_deductions)
        rules = sorted(by_rule)
        while remaining and rules:
            active = []
            for rule in rules:
                if by_rule[rule]:
                    selected.append(by_rule[rule].pop())
                    remaining -= 1
                    if by_rule[rule]:
                        active.append(rule)
                    if remaining == 0:
                        break
            rules = active
        selected.extend(r for r in rare if r["step_index"] not in start_steps)
    selected.sort(key=lambda r: hashlib.sha256(
        f"{r['puzzle_id']}:{r['step_index']}:shuffle".encode()).hexdigest())
    return selected


In [ ]:
%%writefile /kaggle/working/tt/tt_lib.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Safe bounded search for Train Tracks branch values.

The evaluator offers a branch only when no deduction remains, and only two
values of one cell.  A wrong value can cost dozens of forced deductions before
a contradiction, so we spend a little CPU here to reject values that are
provably impossible under the same rule engine.

The filter is *safe*: it only rejects a value when propagation plus a bounded
search reaches a contradiction.  If the current board still contains the true
solution, the true value cannot be rejected.  When both values survive the
search, callers should fall back to the model's choice.

Validated on the run-03 medium rollouts: on boards that still contain the
reference solution, depth 2 raised branch accuracy from 68.2% to 92.4% and never
rejected the true value.  Depth 3 added nothing, so depth 2 is the default.
"""

from __future__ import annotations

from tt_core import legal_actions
from tt_core import BIT, TILES, branch_cell, choices, completed_error

DEFAULT_DEPTH = 2


def clue_from_state(state: dict) -> dict:
    return {
        "rows": state["row_clues"],
        "columns": state["column_clues"],
        "given_shapes": [
            (g["row"] - 1, g["column"] - 1, g["value"]) for g in state["givens"]
        ],
    }


def _mask(candidates) -> int:
    return sum(BIT[c] for c in candidates)


def propagate(board, state: dict, stack: list, round_limit: int = 400):
    """Apply every currently legal deduction until a fixpoint or contradiction.

    Returns ``(contradicted, board, deductions)``.  Actions that became stale
    after an earlier action in the same round are skipped rather than treated as
    contradictions.
    """
    board = [row[:] for row in board]
    deductions = 0
    for _ in range(round_limit):
        local = {
            "row_clues": state["row_clues"],
            "column_clues": state["column_clues"],
            "givens": state["givens"],
            "domains": board,
            "branch_stack": stack,
        }
        actions = legal_actions(local, pending=False)
        if any(a["type"] == "contradiction" for a in actions):
            return True, board, deductions
        deduces = [a for a in actions if a["type"] == "deduce"]
        if not deduces:
            return False, board, deductions
        applied = 0
        for action in deduces:
            r = action["cell"]["row"] - 1
            c = action["cell"]["column"] - 1
            new = _mask(action["candidates"])
            if new == board[r][c] or new & board[r][c] != new:
                continue
            board[r][c] = new
            deductions += 1
            applied += 1
        if not applied:
            return False, board, deductions
    return False, board, deductions


def possible(board, state: dict, stack: list, depth: int, node_budget: list | None = None) -> bool:
    """Return True if the board is not proven impossible within ``depth`` guesses."""
    if node_budget is not None:
        if node_budget[0] <= 0:
            return True
        node_budget[0] -= 1
    contradicted, board, _ = propagate(board, state, stack)
    if contradicted:
        return False
    clue = clue_from_state(state)
    if branch_cell(board) is None:
        return completed_error(board, clue) is None
    if depth <= 0:
        return True  # unknown: conservatively possible
    r, c = branch_cell(board)
    cell = {"row": r + 1, "column": c + 1}
    for value in choices(board[r][c]):
        child = [row[:] for row in board]
        child[r][c] = BIT[value]
        child_stack = list(stack) + [
            {"to_step": -1, "cell": cell, "value": value}
        ]
        if possible(child, state, child_stack, depth - 1, node_budget):
            return True
    return False


def surviving_values(state: dict, cell: dict, depth: int = DEFAULT_DEPTH,
                     node_budget: int = 4000) -> dict:
    """Probe both values at ``cell`` and return details about survivors."""
    r, c = cell["row"] - 1, cell["column"] - 1
    candidates = [t for t in TILES if state["domains"][r][c] & BIT[t]]
    budget = [node_budget]
    feasible = {}
    for value in candidates:
        child = [row[:] for row in state["domains"]]
        child[r][c] = BIT[value]
        child_stack = list(state["branch_stack"]) + [
            {"to_step": -1, "cell": cell, "value": value}
        ]
        feasible[value] = possible(child, state, child_stack, depth, budget)
    survivors = [v for v in candidates if feasible[v]]
    return {"candidates": candidates, "feasible": feasible, "survivors": survivors,
            "depth": depth, "nodes_left": budget[0]}


def choose_branch(state: dict, cell: dict, model_value: str,
                  depth: int = DEFAULT_DEPTH, node_budget: int = 4000) -> dict:
    """Pick a branch value, using the model choice when it survives.

    Returns a dict with the chosen ``value``, whether the search overrode the
    model, and the survivor details for logging.
    """
    if depth <= 0:
        return {"value": model_value, "override": False, "search": None}
    info = surviving_values(state, cell, depth=depth, node_budget=node_budget)
    survivors = info["survivors"]
    if len(survivors) == 1:
        value = survivors[0]
    elif model_value in survivors:
        value = model_value
    elif survivors:
        value = survivors[0]
    else:
        value = model_value  # poisoned board: the filter cannot help
    return {"value": value, "override": value != model_value, "search": info}


"""Build the Kaggle training views, including branch-focused augmentation.

Two changes relative to the original Modal baseline:

1. **Every branch decision is trained in both menu orders.**  The evaluator
   sorts the legal menu, so a model can learn "pick a001" instead of reading the
   board.  Reversing the two branch entries removes that shortcut.
2. **Wrong-branch trace rows are reused as corrected positives.**  The original
   `split_examples` drops rows with ``expert_policy_label: false``.  Their state
   is still a valid pre-branch state, and the audited solution tells us the
   correct value.  Adding that corrected example gives the model the hard states
   where the teacher probe guessed wrong, without ever training a wrong target.

Branch examples are optionally repeated so they are not drowned by the ~8.6k
deduction examples.
"""

import argparse
import json
import random
from collections import Counter
from pathlib import Path

from tt_core import (choice_id, completion, development_ids,
                           legal_actions, prompt, split_examples)

HERE = Path(__file__).resolve().parent
TILES = ("XX", "NS", "EW", "NE", "NW", "SE", "SW")


def load_jsonl(path: Path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle]


def load_solutions(path: Path) -> dict[str, list[list[str]]]:
    solutions = {}
    for record in load_jsonl(path):
        solutions[record["puzzle_id"]] = [
            [tile or "XX" for tile in row] for row in record["solution"]
        ]
    return solutions


def _make_example(row, actions, target):
    return {
        "user": prompt(row["state"], actions),
        "completion": completion(target),
        "kind": row["action"]["type"],
        "puzzle_id": row["puzzle_id"],
        "target": target,
        "menu_size": len(actions),
    }


def _branch_pair(mask: int) -> str:
    return "|".join(name for index, name in enumerate(TILES) if mask & (1 << index))


def _corrected_branch_row(row, solution):
    cell = row["action"]["cell"]
    correct = solution[cell["row"] - 1][cell["column"] - 1]
    actions = legal_actions(row["state"], pending=False)
    branch_actions = [a for a in actions if a["type"] == "branch"]
    if not any(a["cell"] == cell and a["value"] == correct for a in branch_actions):
        return None, None
    corrected = {"type": "branch", "cell": cell, "value": correct}
    corrected_row = dict(row)
    corrected_row["action"] = corrected
    return corrected_row, actions


def build_examples(rows, solutions=None, *, branch_repeat: int = 2,
                   reverse_branch_orders: bool = True,
                   corrected_wrong_branches: bool = True,
                   max_deductions_per_puzzle: int = 48,
                   seed: int = 42):
    """Return ``(examples, stats)`` for a list of trace rows."""
    selected = split_examples(rows, max_deductions_per_puzzle)
    examples = []
    by_kind = Counter()
    pairs = Counter()
    values = Counter()
    corrected = 0
    reversed_orders = 0

    def add(row, actions, target):
        examples.append(_make_example(row, actions, target))

    for row in selected:
        pending = row["action"]["type"] == "undo"
        actions = legal_actions(row["state"], pending=pending)
        target = choice_id(row["action"], actions)
        add(row, actions, target)
        by_kind[row["action"]["type"]] += 1
        if row["action"]["type"] == "branch":
            cell = row["action"]["cell"]
            pairs[_branch_pair(row["state"]["domains"][cell["row"] - 1][cell["column"] - 1])] += 1
            values[row["action"]["value"]] += 1
            if reverse_branch_orders:
                rev = list(reversed(actions))
                add(row, rev, choice_id(row["action"], rev))
                reversed_orders += 1

    if corrected_wrong_branches and solutions is not None:
        for row in rows:
            if row["action"]["type"] != "branch" or row.get("expert_policy_label", True):
                continue
            solution = solutions.get(row["puzzle_id"])
            if solution is None:
                continue
            corrected_row, actions = _corrected_branch_row(row, solution)
            if corrected_row is None:
                continue
            target = choice_id(corrected_row["action"], actions)
            add(corrected_row, actions, target)
            by_kind["branch_corrected"] += 1
            cell = corrected_row["action"]["cell"]
            pairs[_branch_pair(corrected_row["state"]["domains"][cell["row"] - 1][cell["column"] - 1])] += 1
            values[corrected_row["action"]["value"]] += 1
            corrected += 1
            if reverse_branch_orders:
                rev = list(reversed(actions))
                add(corrected_row, rev, choice_id(corrected_row["action"], rev))
                reversed_orders += 1

    if branch_repeat > 1:
        branch_examples = [
            e for e in examples if e["kind"] in {"branch", "branch_corrected"}
        ]
        for _ in range(branch_repeat - 1):
            examples.extend(branch_examples)

    rng = random.Random(seed)
    rng.shuffle(examples)
    stats = {
        "total": len(examples),
        "base_selected": len(selected),
        "by_kind": dict(by_kind),
        "corrected_wrong_branches": corrected,
        "reversed_branch_orders": reversed_orders,
        "branch_repeat": branch_repeat,
        "branch_examples_after_repeat": sum(
            1 for e in examples if e["kind"] in {"branch", "branch_corrected"}),
        "branch_pair_counts": dict(pairs),
        "branch_value_counts": dict(values),
    }
    return examples, stats


def build_dev_examples(rows, max_deductions_per_puzzle: int = 48):
    """Clean dev view: base examples only, no augmentation."""
    selected = split_examples(rows, max_deductions_per_puzzle)
    examples = []
    for row in selected:
        actions = legal_actions(row["state"], pending=row["action"]["type"] == "undo")
        target = choice_id(row["action"], actions)
        examples.append(_make_example(row, actions, target))
    return examples


def split_rows(rows):
    dev_ids = development_ids(row["puzzle_id"] for row in rows)
    train_rows = [row for row in rows if row["puzzle_id"] not in dev_ids]
    dev_rows = [row for row in rows if row["puzzle_id"] in dev_ids]
    return train_rows, dev_rows, len(dev_ids)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, default=HERE / "data/train.jsonl")
    parser.add_argument("--source", type=Path,
                        default=HERE / "data/train-puzzles-small.jsonl")
    parser.add_argument("--branch-repeat", type=int, default=2)
    parser.add_argument("--no-branch-augment", action="store_true")
    parser.add_argument("--write", type=Path, default=None,
                        help="Optional JSONL output with user/completion fields")
    args = parser.parse_args()

    rows = load_jsonl(args.data)
    solutions = load_solutions(args.source)
    train_rows, dev_rows, dev_count = split_rows(rows)
    if args.no_branch_augment:
        examples, stats = build_examples(
            train_rows, solutions, branch_repeat=1,
            reverse_branch_orders=False, corrected_wrong_branches=False)
    else:
        examples, stats = build_examples(
            train_rows, solutions, branch_repeat=args.branch_repeat)
    dev = build_dev_examples(dev_rows)
    stats["puzzles_total"] = len({r["puzzle_id"] for r in rows})
    stats["puzzles_train"] = len({r["puzzle_id"] for r in train_rows})
    stats["puzzles_dev"] = dev_count
    stats["dev_examples"] = len(dev)
    print(json.dumps(stats, indent=2))
    if args.write:
        with args.write.open("w", encoding="utf-8", newline="\n") as handle:
            for example in examples:
                handle.write(json.dumps(example, ensure_ascii=False) + "\n")
        print(f"wrote {len(examples)} examples to {args.write}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tt/tt_prep.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Build branch-state SFT examples from the audited medium solutions.

Medium boards are 9x9..10x10 and the run-04 adapter guesses the wrong branch on
roughly half of the hard ones, so this module turns the audited solutions into
supervised branch decisions for the *training* part of the medium pool.

Two properties matter:

* **Cheap.**  Medium rollouts are generated on CPU by an oracle that plays the
  audited solution, so a full trace costs under a second.  Only branch states
  plus a small sample of deductions are kept (~30 examples per puzzle instead of
  the ~170-step trace), which puts a complete 240-puzzle training set at
  **7,488 rows / ~6.9M tokens** (measured, 2.7 min on a laptop CPU).
* **Leak-free.**  ``build_pool`` generates examples for whatever records it is
  given and never chooses ids.  The one place ids are chosen is ``split_pool``,
  and it reuses the evaluator's stratified sampler so the evaluation ids are
  exactly the 32 run-04 scored, with a second held-out block left untouched and
  only the remainder trained on.
"""

import argparse
import json
from pathlib import Path

from tt_core import choice_id, completion
from tt_lib import build_examples
from tt_eval import rollout, select_medium_sample

# The 32 puzzles run-04 evaluated at 256 actions.  They are the evaluation set
# and must never appear in training; the constant only drives the assertion in
# `split_pool`, the selection itself comes from the evaluator's sampler.
RUN04_EVAL_IDS = [
    "01012026", "02022026", "03032026", "03092026", "04062026", "05022026",
    "05082026", "06052026", "08102026", "10012026", "12062026", "13072026",
    "14072026", "17042026", "18102026", "19042026", "19062026", "19102026",
    "20012026", "20062026", "22012026", "22022026", "24032026", "24102026",
    "25022026", "27012026", "28052026", "28102026", "29052026", "29082026",
    "30032026", "30082026",
]


def reference_board(record) -> list[list[str]]:
    return [[tile or "XX" for tile in row] for row in record["solution"]]


def oracle_action(actions, reference):
    """Play the audited solution: undo when pending, else take the true value."""
    kinds = {action["type"] for action in actions}
    if "undo" in kinds:
        return next(action for action in actions if action["type"] == "undo")
    if "contradiction" in kinds:
        raise AssertionError(f"clean board produced a contradiction: {actions}")
    if "finish" in kinds:
        return {"type": "finish"}
    if "branch" in kinds:
        branches = [action for action in actions if action["type"] == "branch"]
        cell = branches[0]["cell"]
        truth = reference[cell["row"] - 1][cell["column"] - 1]
        return next((a for a in branches if a["value"] == truth), branches[0])
    return next(action for action in actions if action["type"] == "deduce")


def oracle_rows(record, max_actions: int = 512):
    """Replay the audited solution and return ``(rows, result)``.

    Each row is ``{puzzle_id, step_index, state, action, expert_policy_label}``,
    the format ``data_prep.build_examples`` consumes.
    """
    reference = reference_board(record)
    rows = []

    def policy(state, actions):
        action = oracle_action(actions, reference)
        rows.append({
            "puzzle_id": record["puzzle_id"],
            "step_index": len(rows),
            "state": state,
            "action": action,
            "expert_policy_label": True,
        })
        return completion(choice_id(action, actions))

    args = type("TraceArgs", (), {
        "max_actions": max_actions,
        "branch_search_depth": 0,
        "branch_search_budget": 1,
        "max_new_tokens": 16,
        "compact": True,
    })()
    result = rollout(policy, record, args)
    return rows, result


def puzzle_examples(record, *, max_deductions_per_puzzle: int = 8,
                    branch_repeat: int = 1, max_actions: int = 512):
    """Return ``(examples, stats, result)`` for one puzzle."""
    rows, result = oracle_rows(record, max_actions=max_actions)
    if not result["solved"]:
        raise AssertionError(
            f"{record['puzzle_id']}: oracle rollout failed ({result['failure']})")
    examples, stats = build_examples(
        rows, {record["puzzle_id"]: reference_board(record)},
        branch_repeat=branch_repeat,
        max_deductions_per_puzzle=max_deductions_per_puzzle)
    for example in examples:
        example["puzzle_id"] = record["puzzle_id"]
    return examples, stats, result


def build_pool(records, *, max_deductions_per_puzzle: int = 8,
               branch_repeat: int = 1, max_actions: int = 512, log=None):
    """Build examples for a list of puzzle records, one puzzle at a time.

    Returns ``(examples, report)`` where the report carries per-puzzle counts so
    a collapsed generation (e.g. an oracle failure) is visible immediately.
    """
    examples = []
    report = {"puzzles": 0, "oracle_actions": 0, "branch_examples": 0,
              "deduce_examples": 0, "by_kind": {}, "puzzles_with_short_trace": []}
    for index, record in enumerate(records, 1):
        built, stats, result = puzzle_examples(
            record, max_deductions_per_puzzle=max_deductions_per_puzzle,
            branch_repeat=branch_repeat, max_actions=max_actions)
        examples.extend(built)
        report["puzzles"] += 1
        report["oracle_actions"] += result["actions"]
        report["branch_examples"] += sum(
            1 for example in built
            if example["kind"] in {"branch", "branch_corrected"})
        report["deduce_examples"] += sum(
            1 for example in built if example["kind"] == "deduce")
        for kind, count in stats["by_kind"].items():
            report["by_kind"][kind] = report["by_kind"].get(kind, 0) + count
        if result["actions"] < 80:
            report["puzzles_with_short_trace"].append(record["puzzle_id"])
        if log and (index % 20 == 0 or index == len(records)):
            log(f"generated {index}/{len(records)} puzzles, "
                f"{len(examples):,} examples")
    return examples, report


def write_jsonl(path, examples):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        for example in examples:
            handle.write(json.dumps(example, ensure_ascii=False) + "\n")
    return len(examples)


def split_pool(records, eval_count: int = 32, fresh_count: int = 32):
    """Split the medium pool into ``(train, eval, fresh)`` record lists.

    ``eval`` is the block run-04 measured, so the comparison is like-for-like;
    ``fresh`` is a second untouched block used as development loss; ``train`` is
    the remainder.  The three are disjoint and cover the input.
    """
    records = sorted(records, key=lambda record: record["puzzle_id"])
    eval_records = select_medium_sample(records, eval_count)
    eval_ids = {record["puzzle_id"] for record in eval_records}
    expected = {f"{pid}-medium" for pid in RUN04_EVAL_IDS}
    if eval_ids != expected:
        extra = sorted(eval_ids - expected)
        missing = sorted(expected - eval_ids)
        raise AssertionError(
            f"medium split drifted: {len(eval_ids)} eval ids, "
            f"only-in-input {extra[:5]}, only-in-run04 {missing[:5]}")

    remaining = [record for record in records if record["puzzle_id"] not in eval_ids]
    fresh_records = select_medium_sample(remaining, fresh_count)
    fresh_ids = {record["puzzle_id"] for record in fresh_records}
    train_records = [record for record in remaining
                     if record["puzzle_id"] not in fresh_ids]
    if eval_ids & fresh_ids or len(train_records) + len(eval_ids) + len(fresh_ids) != len(records):
        raise AssertionError("medium split is not disjoint")
    return train_records, eval_records, fresh_records


def estimate_tokens(examples, chars_per_token: float = 3.6) -> int:
    """Rough token count without loading a tokenizer."""
    chars = sum(len(example["user"]) + len(example["completion"])
                for example in examples)
    return int(chars / chars_per_token)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, required=True,
                        help="train-puzzles-medium.jsonl from the data dataset")
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--max-deductions", type=int, default=16,
                        help="Deduction examples kept per training puzzle")
    parser.add_argument("--dev-deductions", type=int, default=4,
                        help="Deduction examples kept per held-out puzzle")
    parser.add_argument("--branch-repeat", type=int, default=1)
    parser.add_argument("--max-actions", type=int, default=512)
    parser.add_argument("--limit", type=int, default=0,
                        help="Cap each pool for a smoke run (the split check still runs)")
    args = parser.parse_args()

    records = [json.loads(line) for line in args.data.open(encoding="utf-8")]
    medium = [record for record in records if record.get("split") == "test_medium"]
    print(f"medium puzzles: {len(medium)} (of {len(records)} records)")

    train_records, eval_records, fresh_records = split_pool(medium)
    print(f"split: train {len(train_records)} | eval {len(eval_records)} | "
          f"fresh {len(fresh_records)}")
    if args.limit:
        train_records = train_records[:args.limit]
        fresh_records = fresh_records[:args.limit]
        eval_records = eval_records[:args.limit]
        print(f"limit applied: {args.limit} puzzles per pool")

    def log(message):
        print(message, flush=True)

    train_examples, train_report = build_pool(
        train_records, max_deductions_per_puzzle=args.max_deductions,
        branch_repeat=args.branch_repeat, max_actions=args.max_actions, log=log)
    dev_examples, dev_report = build_pool(
        fresh_records, max_deductions_per_puzzle=args.dev_deductions,
        branch_repeat=1, max_actions=args.max_actions, log=log)

    train_path = args.output_dir / "train-examples.jsonl"
    dev_path = args.output_dir / "dev-examples.jsonl"
    write_jsonl(train_path, train_examples)
    write_jsonl(dev_path, dev_examples)

    report = {
        "data": str(args.data),
        "max_actions": args.max_actions,
        "max_deductions_per_puzzle": args.max_deductions,
        "branch_repeat": args.branch_repeat,
        "split": {
            "train": len(train_records),
            "eval": len(eval_records),
            "fresh": len(fresh_records),
            "eval_ids": [record["puzzle_id"] for record in eval_records],
            "fresh_ids": [record["puzzle_id"] for record in fresh_records],
            "matches_run04": not args.limit,
        },
        "examples": {"train": len(train_examples), "dev": len(dev_examples)},
        "estimated_tokens": {
            "train": estimate_tokens(train_examples),
            "dev": estimate_tokens(dev_examples),
        },
        "train_report": train_report,
        "dev_report": dev_report,
        "files": {"train": str(train_path), "dev": str(dev_path)},
    }
    report_path = args.output_dir / "generation-report.json"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(json.dumps(report, indent=1) + "\n", encoding="utf-8")

    print(f"\ntrain examples: {len(train_examples):,} "
          f"(~{report['estimated_tokens']['train'] / 1e6:.1f}M tokens)")
    print(f"dev examples:   {len(dev_examples):,}")
    print(f"branch examples: {train_report['branch_examples']:,} | "
          f"deduce: {train_report['deduce_examples']:,}")
    print(f"oracle actions: {train_report['oracle_actions']:,}")
    print(f"wrote {train_path}")
    print(f"wrote {dev_path}")
    print(f"wrote {report_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tt/tt_dagger.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Collect the policy's own branch states and label them from the solution.

Run-05 trained on *teacher* states — boards the oracle reaches — and cut dev loss
58% without changing a single branch decision on the puzzles that fail. The model
is already right on the boards the teacher visits, and wrong on the boards it
creates itself. This module records those boards (DAgger).

Two things are stored per branch decision:

* the exact board the policy saw, so the training prompt is byte-identical to the
  one the evaluator would build;
* whether the audited value was still in the menu. A poisoned board can have
  eliminated the true tile; those states cannot be labelled and are counted
  separately rather than silently dropped or, worse, trained with the other value.

The collector wraps any policy callable, so the same code runs against an adapter
on Kaggle and against the CPU oracle used by the tests.
"""

import argparse
import copy
import json
from pathlib import Path

from tt_lib import build_examples
from tt_eval import load_policy, rollout
from tt_prep import reference_board, write_jsonl


def branch_action(actions):
    """Return the branch action offered by this menu, or ``None``."""
    return next((action for action in actions if action["type"] == "branch"), None)


def collect(policy, record, args, log=None):
    """Run one puzzle under ``policy`` and record every branch state it sees.

    Returns ``(states, result)`` where each state is a JSON-ready dict.  States are
    keyed by the step index the harness used, so a refused/invalid reply still
    lines up with the event it produced.
    """
    truth_board = reference_board(record)
    states = []
    counter = {"step": 0}

    def recorder(state, actions):
        step = counter["step"]
        counter["step"] += 1
        branch = branch_action(actions)
        if branch is not None:
            cell = branch["cell"]
            truth = truth_board[cell["row"] - 1][cell["column"] - 1]
            offered = [action["value"] for action in actions
                       if action["type"] == "branch" and action["cell"] == cell]
            states.append({
                "puzzle_id": record["puzzle_id"],
                "step": step,
                "cell": cell,
                "truth": truth,
                "offered": offered,
                "recoverable": truth in offered,
                "state": copy.deepcopy(state),
            })
        return policy(state, actions)

    result = rollout(recorder, record, args)

    # Fill in what the policy actually committed at each of those states.
    events = {event["step"]: event for event in result["history"]}
    for item in states:
        event = events.get(item["step"]) or {}
        action = event.get("action") or {}
        item["model"] = action.get("value") if action.get("type") == "branch" else None
        item["override"] = bool((event.get("branch_search") or {}).get("override"))

    if log:
        correct = sum(item["model"] == item["truth"] for item in states)
        log(f"  {record['puzzle_id']:18s} branches {len(states):2d} "
            f"correct {correct:2d} unrecoverable "
            f"{sum(not item['recoverable'] for item in states):2d} "
            f"actions {result['actions']:3d} "
            f"{'solved' if result['solved'] else 'failed'}")
    return states, result


def dedupe(states):
    """Drop repeated boards: a revisit after undo is the same training prompt."""
    seen, unique, repeated = set(), [], 0
    for item in states:
        key = (item["puzzle_id"], json.dumps(item["state"], sort_keys=True))
        if key in seen:
            repeated += 1
            continue
        seen.add(key)
        unique.append(item)
    return unique, repeated


def build_from_states(states, *, branch_repeat: int = 1, reverse_orders: bool = True):
    """Turn recorded states into ``user``/``completion`` rows.

    Every row's target is the audited value, so a wrong model choice becomes a
    corrected positive.  States where the truth was no longer offered are counted
    and skipped: there is nothing to teach on a board that cannot reach the
    solution.
    """
    unique, repeated = dedupe(states)
    rows = []
    stats = {
        "states": len(states),
        "repeated_boards": repeated,
        "unique_states": len(unique),
        "model_correct": sum(item["model"] == item["truth"] for item in unique),
        "model_wrong": sum(item["model"] != item["truth"] for item in unique),
        "unrecoverable": sum(not item["recoverable"] for item in unique),
        "puzzles": len({item["puzzle_id"] for item in unique}),
    }
    for item in unique:
        if not item["recoverable"]:
            continue
        rows.append({
            "puzzle_id": item["puzzle_id"],
            "step_index": item["step"],
            "state": item["state"],
            "action": {"type": "branch", "cell": item["cell"], "value": item["truth"]},
            "expert_policy_label": True,
        })

    examples, build_stats = build_examples(
        rows, None, branch_repeat=branch_repeat,
        reverse_branch_orders=reverse_orders,
        corrected_wrong_branches=False,
        max_deductions_per_puzzle=0)
    stats["labelled_rows"] = len(rows)
    stats["examples"] = len(examples)
    stats["by_kind"] = build_stats["by_kind"]
    stats["estimated_tokens"] = int(
        sum(len(example["user"]) + len(example["completion"]) for example in examples) / 3.6)
    return examples, stats


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--adapter", type=Path, default=None)
    parser.add_argument("--data", type=Path, default=None,
                        help="train-puzzles-medium.jsonl; with --build it is only used for a split check")
    parser.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--only", default=None,
                        help="Comma-separated ids to roll out")
    parser.add_argument("--limit", type=int, default=0,
                        help="First N of --only, for a smoke run")
    parser.add_argument("--max-actions", type=int, default=256)
    parser.add_argument("--max-new-tokens", type=int, default=16)
    parser.add_argument("--branch-search-depth", type=int, default=2)
    parser.add_argument("--branch-search-budget", type=int, default=4000)
    parser.add_argument("--output", type=Path, default=None,
                        help="JSONL of recorded states (collection mode)")
    parser.add_argument("--local-files-only", action="store_true")
    parser.add_argument("--build", action="store_true",
                        help="Build examples from --states instead of running the model")
    parser.add_argument("--states", default=None,
                        help="Comma-separated state JSONL files for --build")
    parser.add_argument("--output-dir", type=Path, default=None)
    parser.add_argument("--role", choices=("train", "dev"), default="train")
    parser.add_argument("--branch-repeat", type=int, default=1)
    return parser.parse_args()


def main():
    args = parse_args()

    if args.build:
        if not args.states or not args.output_dir:
            raise SystemExit("--build needs --states and --output-dir")
        states = []
        for name in [part.strip() for part in args.states.split(",") if part.strip()]:
            path = Path(name)
            with path.open(encoding="utf-8") as handle:
                chunk = [json.loads(line) for line in handle]
            print(f"{path.name}: {len(chunk)} states")
            states.extend(chunk)
        if args.only:
            wanted = {part.strip() for part in args.only.split(",") if part.strip()}
            states = [item for item in states if item["puzzle_id"] in wanted]
            print(f"filtered to {len(states)} states in {len(wanted)} puzzles")
        examples, stats = build_from_states(
            states, branch_repeat=1 if args.role == "dev" else args.branch_repeat,
            reverse_orders=args.role != "dev")
        args.output_dir.mkdir(parents=True, exist_ok=True)
        out = args.output_dir / f"{args.role}-examples.jsonl"
        write_jsonl(out, examples)
        report = {"role": args.role, "states": len(states), **stats, "file": str(out)}
        (args.output_dir / f"{args.role}-report.json").write_text(
            json.dumps(report, indent=1) + "\n", encoding="utf-8")
        print(json.dumps({key: report[key] for key in
                          ("role", "states", "unique_states", "model_correct",
                           "model_wrong", "unrecoverable", "labelled_rows",
                           "examples", "estimated_tokens")}, indent=1))
        print(f"wrote {out}")
        return

    if not args.adapter or not args.data or not args.output:
        raise SystemExit("collection needs --adapter, --data and --output")
    records = [json.loads(line) for line in args.data.open(encoding="utf-8")]
    by_id = {record["puzzle_id"]: record for record in records}
    wanted = [part.strip() for part in (args.only or "").split(",") if part.strip()]
    if not wanted:
        raise SystemExit("--only is required for collection")
    missing = [pid for pid in wanted if pid not in by_id]
    if missing:
        raise SystemExit(f"--only: not in this data file: {missing}")
    if args.limit:
        wanted = wanted[:args.limit]

    policy = load_policy(args)
    rollout_args = argparse.Namespace(
        max_actions=args.max_actions, branch_search_depth=args.branch_search_depth,
        branch_search_budget=args.branch_search_budget,
        max_new_tokens=args.max_new_tokens, compact=True)

    all_states = []
    for index, puzzle_id in enumerate(wanted, 1):
        states, _ = collect(policy, by_id[puzzle_id], rollout_args,
                            log=lambda message, i=index: print(
                                f"[{i}/{len(wanted)}]{message}", flush=True))
        all_states.extend(states)

    args.output.parent.mkdir(parents=True, exist_ok=True)
    with args.output.open("w", encoding="utf-8", newline="\n") as handle:
        for item in all_states:
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")
    summary = {
        "output": str(args.output),
        "puzzles": len(wanted),
        "states": len(all_states),
        "model_correct": sum(item["model"] == item["truth"] for item in all_states),
        "unrecoverable": sum(not item["recoverable"] for item in all_states),
        "overrides": sum(item["override"] for item in all_states),
    }
    print(json.dumps(summary, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tt/tt_train.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Train the branch-focused Qwen2.5-1.5B action policy on Kaggle T4 x2.

This is the Kaggle port of ``modal/train.py`` with three changes:

* no Modal dependency and local data paths;
* fp16 on T4 (bf16 only when the GPU supports it);
* branch-focused data augmentation from ``data_prep.py`` (both menu orders,
  corrected wrong-branch states, branch upsampling).

Run from the ``kaggle/`` directory:

    python train_kaggle.py --output /kaggle/working/run-branch

For 2-GPU DDP:

    accelerate launch --config_file configs/accelerate_t4x2.yaml train_kaggle.py ...
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

from tt_core import SYSTEM  # noqa: E402
from tt_lib import (build_dev_examples, build_examples, load_jsonl,
                       load_solutions, split_rows)  # noqa: E402


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, default=HERE / "data/train.jsonl")
    parser.add_argument("--source", type=Path,
                        default=HERE / "data/train-puzzles-small.jsonl")
    parser.add_argument("--examples", type=Path, default=None,
                        help="Precomputed user/completion JSONL; skips --data/--source")
    parser.add_argument("--dev-examples", type=Path, default=None,
                        help="Precomputed dev JSONL paired with --examples")
    parser.add_argument("--init-adapter", type=Path, default=None,
                        help="Continue training this LoRA adapter instead of adding a new one")
    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/run-branch"))
    parser.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--revision", default="main")
    parser.add_argument("--epochs", type=float, default=1.0)
    parser.add_argument("--max-steps", type=int, default=-1)
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--grad-accum", type=int, default=8)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--max-length", type=int, default=2048)
    parser.add_argument("--branch-repeat", type=int, default=2)
    parser.add_argument("--no-branch-augment", action="store_true")
    parser.add_argument("--smoke", action="store_true",
                        help="16 train / 8 dev examples, 4 optimizer steps")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--local-files-only", action="store_true")
    return parser.parse_args()


def load_model_and_tokenizer(args):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    # With --init-adapter the adapter directory also holds the tokenizer that
    # rendered its training prompts, so the chat template matches exactly.
    tokenizer_source = args.model
    if args.init_adapter and (args.init_adapter / "tokenizer_config.json").exists():
        tokenizer_source = args.init_adapter
    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source, revision=None if tokenizer_source != args.model else args.revision,
        local_files_only=args.local_files_only)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # bf16 requires sm_80+.  `is_bf16_supported()` can return True on some T4
    # images, where bf16 then falls back to slow emulation, so check the
    # capability explicitly.
    capability = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
    bf16_ok = capability[0] >= 8 and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16_ok else torch.float16
    kwargs = dict(revision=args.revision, local_files_only=args.local_files_only,
                  attn_implementation="sdpa")
    try:
        model = AutoModelForCausalLM.from_pretrained(args.model, dtype=dtype, **kwargs)
    except TypeError:  # older transformers
        model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=dtype, **kwargs)
    if args.init_adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(args.init_adapter), is_trainable=True)
        # Gradient checkpointing recomputes forward passes; some transformers/PEFT
        # combinations need the embedding output to require grad for LoRA grads.
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
    model.config.use_cache = False
    return model, tokenizer, dtype


def to_dataset(examples, tokenizer):
    from datasets import Dataset

    rows = []
    for example in examples:
        prompt = tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM},
             {"role": "user", "content": example["user"]}],
            tokenize=False, add_generation_prompt=True)
        rows.append({
            "prompt": prompt,
            "completion": example["completion"] + tokenizer.eos_token,
            "kind": example["kind"],
        })
    return Dataset.from_list(rows)


def sft_config(args, output_dir, fp16, bf16):
    from trl import SFTConfig

    fields = set(SFTConfig.__dataclass_fields__)
    kwargs = {
        "output_dir": str(output_dir),
        "num_train_epochs": args.epochs,
        "max_steps": 4 if args.smoke else args.max_steps,
        "per_device_train_batch_size": args.batch_size,
        "gradient_accumulation_steps": args.grad_accum,
        "per_device_eval_batch_size": max(1, args.batch_size),
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "learning_rate": args.lr,
        "lr_scheduler_type": "cosine",
        "warmup_steps": 1 if args.smoke else 10,
        "logging_steps": 1 if args.smoke else 10,
        "eval_strategy": "no",
        "save_strategy": "no",
        "bf16": bf16,
        "fp16": fp16,
        "report_to": "none",
        "completion_only_loss": True,
        "packing": False,
        "max_length": args.max_length,
        "seed": args.seed,
        "data_seed": args.seed,
        "dataloader_num_workers": 2,
    }
    if "max_seq_length" in fields and "max_length" not in fields:
        kwargs["max_seq_length"] = kwargs.pop("max_length")
    if "eval_strategy" not in fields and "evaluation_strategy" in fields:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    kwargs = {key: value for key, value in kwargs.items() if key in fields}
    return SFTConfig(**kwargs)


def pin_single_gpu_if_unlaunched():
    """Keep TRL's SFTTrainer out of ``nn.DataParallel``.

    A single process that can see several GPUs makes ``Trainer`` wrap the model
    in ``nn.DataParallel``, and ``SFTTrainer`` refuses that outright.  Launchers
    (``accelerate launch``, ``torchrun``) set ``WORLD_SIZE`` and assign devices
    themselves; for a plain run we pin the process to GPU 0.

    This has to happen before the first CUDA call: ``cudaGetDeviceCount``
    caches the visible-device list, so setting the variable afterwards is too
    late.  That is why there is no device query here.
    """
    if int(os.environ.get("WORLD_SIZE", "1")) > 1:
        return
    if os.environ.get("CUDA_VISIBLE_DEVICES") is not None:
        return
    print("single-process run: pinning to GPU 0 "
          "(SFTTrainer cannot use nn.DataParallel; use accelerate launch for 2 GPUs)",
          flush=True)
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"


def main():
    args = parse_args()
    if args.branch_repeat < 1:
        raise SystemExit("--branch-repeat must be at least 1")
    pin_single_gpu_if_unlaunched()
    import torch
    from peft import LoraConfig
    from transformers import set_seed
    from trl import SFTTrainer

    set_seed(args.seed)
    args.output.mkdir(parents=True, exist_ok=True)

    if args.examples:
        train_examples = load_jsonl(args.examples)
        dev_examples = load_jsonl(args.dev_examples) if args.dev_examples else []
        train_puzzles = len({example.get("puzzle_id") for example in train_examples})
        dev_puzzles = len({example.get("puzzle_id") for example in dev_examples})
        train_rows = []
        stats = {"source": "precomputed", "total": len(train_examples),
                 "puzzles": train_puzzles}
        print(f"precomputed examples: {len(train_examples):,} train, "
              f"{len(dev_examples):,} dev")
    else:
        rows = load_jsonl(args.data)
        solutions = load_solutions(args.source)
        train_rows, dev_rows, dev_puzzles = split_rows(rows)
        if args.no_branch_augment:
            train_examples, stats = build_examples(
                train_rows, solutions, branch_repeat=1,
                reverse_branch_orders=False, corrected_wrong_branches=False)
        else:
            train_examples, stats = build_examples(
                train_rows, solutions, branch_repeat=args.branch_repeat)
        dev_examples = build_dev_examples(dev_rows)
        train_puzzles = len({row["puzzle_id"] for row in train_rows})
    if args.smoke:
        train_examples = train_examples[:16]
        dev_examples = dev_examples[:8]

    model, tokenizer, dtype = load_model_and_tokenizer(args)
    train_data = to_dataset(train_examples, tokenizer)
    dev_data = to_dataset(dev_examples, tokenizer)
    print(f"train examples: {len(train_data):,}  dev examples: {len(dev_data):,}")
    print(f"dtype: {dtype}  gpus: {torch.cuda.device_count()}  "
          f"gpu0: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

    fp16 = dtype == torch.float16
    bf16 = dtype == torch.bfloat16
    peft_config = None if args.init_adapter else LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"])
    trainer = SFTTrainer(
        model=model,
        args=sft_config(args, args.output / "trainer", fp16=fp16, bf16=bf16),
        train_dataset=train_data,
        eval_dataset=dev_data,
        peft_config=peft_config,
        processing_class=tokenizer,
    )

    started = time.monotonic()
    before = trainer.evaluate().get("eval_loss")
    trainer.train()
    after = trainer.evaluate().get("eval_loss")
    adapter = args.output / "adapter"

    # Under DDP every process would otherwise write the same files concurrently.
    if trainer.is_world_process_zero():
        trainer.save_model(str(adapter))
        tokenizer.save_pretrained(adapter)
        report = {
            "run_id": args.output.name,
            "model": args.model,
            "task": "trace_action_policy",
            "branch_augment": not args.no_branch_augment,
            "branch_repeat": 1 if args.no_branch_augment else args.branch_repeat,
            "puzzles_train": train_puzzles,
            "puzzles_dev": dev_puzzles,
            "train_examples": len(train_data),
            "dev_examples": len(dev_data),
            "data_stats": stats,
            "examples_file": str(args.examples) if args.examples else None,
            "init_adapter": str(args.init_adapter) if args.init_adapter else None,
            "continued_training": bool(args.init_adapter),
            "optimizer_steps": trainer.state.global_step,
            "development_loss_before": before,
            "development_loss_after": after,
            "train_runtime_seconds": round(time.monotonic() - started, 1),
            "peak_gpu_memory_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
            "dtype": str(dtype),
            "gpus": torch.cuda.device_count(),
            "adapter": str(adapter),
            "output_dir": str(args.output),
        }
        (args.output / "training.json").write_text(json.dumps(report, indent=2) + "\n")
        print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tt/tt_eval.py
# Generated from the verified Kaggle package. Edit this cell, not a file.
"""Evaluate a trained adapter on Kaggle, with an optional safe branch search.

Baseline (model alone):

    python evaluate_kaggle.py --adapter /kaggle/working/run-branch/adapter \
        --branch-search-depth 0 --output baseline.json

Improved (depth-2 branch filter):

    python evaluate_kaggle.py --adapter /kaggle/working/run-branch/adapter \
        --branch-search-depth 2 --output improved.json

The search only rejects a value when propagation proves it impossible, so it
cannot reject the true value on a board that still contains the solution.  When
both values survive, the model's choice is kept.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

from tt_core import (SYSTEM, development_ids, legal_actions,  # noqa: E402
                           parse_action, prompt, validate_action)
from tt_lib import DEFAULT_DEPTH, choose_branch  # noqa: E402
from tt_core import (BIT, clue_from_record, completed_error,  # noqa: E402
                         initial_domains, one, snapshot)


def parse_args():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--adapter", type=Path, required=True)
    parser.add_argument("--data", type=Path,
                        default=HERE / "data/train-puzzles-medium.jsonl")
    parser.add_argument("--small-data", type=Path,
                        default=HERE / "data/train-puzzles-small.jsonl")
    parser.add_argument("--dataset", choices=("medium", "small"), default="medium")
    parser.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--limit", type=int, default=32,
                        help="0 means all; medium uses a shape-stratified sample")
    parser.add_argument("--max-actions", type=int, default=256)
    parser.add_argument("--max-new-tokens", type=int, default=32)
    parser.add_argument("--branch-search-depth", type=int, default=DEFAULT_DEPTH,
                        help="0 disables the branch filter")
    parser.add_argument("--branch-search-budget", type=int, default=4000,
                        help="Max search nodes per branch decision")
    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/eval.json"))
    parser.add_argument("--only", default=None,
                        help="Comma-separated puzzle ids to evaluate; bypasses sampling")
    parser.add_argument("--compact", action="store_true",
                        help="Drop board snapshots from the saved history")
    parser.add_argument("--local-files-only", action="store_true")
    return parser.parse_args()


def state_for(board, clue, stack):
    return {
        "row_clues": clue["rows"], "column_clues": clue["columns"],
        "givens": [{"row": r + 1, "column": c + 1, "value": shape}
                   for r, c, shape in clue["given_shapes"]],
        "domains": [line.copy() for line in board],
        "branch_stack": [{"to_step": item["to_step"], "cell": item["cell"],
                          "value": item["value"]} for item in stack],
    }


def select_medium_sample(puzzles, limit):
    groups = defaultdict(list)
    for puzzle in puzzles:
        shape = (len(puzzle["puzzle"]["rows"]), len(puzzle["puzzle"]["columns"]))
        groups[shape].append(puzzle)
    for group in groups.values():
        group.sort(key=lambda row: hashlib.sha256(row["puzzle_id"].encode()).hexdigest())
    target = min(limit, sum(len(group) for group in groups.values()))
    selected = {shape: [] for shape in groups}
    while sum(len(items) for items in selected.values()) < target:
        eligible = [shape for shape in groups if len(selected[shape]) < len(groups[shape])]
        shape = min(eligible, key=lambda key: (len(selected[key]), key))
        selected[shape].append(groups[shape][len(selected[shape])])
    return sorted((p for items in selected.values() for p in items),
                  key=lambda row: row["puzzle_id"])


def model_policy(model, tokenizer, args):
    """Return ``policy(state, actions) -> raw completion text`` for the adapter."""
    import torch

    def policy(state, actions):
        messages = [{"role": "system", "content": SYSTEM},
                    {"role": "user", "content": prompt(state, actions)}]
        encoded = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=True,
            return_tensors="pt").to("cuda")
        with torch.inference_mode():
            generated = model.generate(
                **encoded, max_new_tokens=args.max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(generated[0, encoded["input_ids"].shape[-1]:],
                                skip_special_tokens=True)

    return policy


def rollout(policy, record, args):
    """Run one puzzle with a ``policy(state, actions) -> raw text`` callable.

    The policy seam keeps the harness testable without a GPU: ``selfcheck.py``
    drives the exact same board updates, branch stack and undo path with an
    oracle policy that reads the audited solution.
    """
    clue = clue_from_record(record)
    board = initial_domains(clue)
    stack, history = [], []
    pending = False
    failure = None
    started = time.monotonic()
    for step_index in range(args.max_actions):
        state = state_for(board, clue, stack)
        actions = legal_actions(state, pending=pending)
        if not actions:
            failure = "no_legal_actions"
            break
        raw = policy(state, actions)
        action = parse_action(raw, actions)
        if action is None:
            valid, reason, canonical = False, "invalid_choice", None
        else:
            valid, reason, canonical = validate_action(board, clue, action, stack, pending)

        model_action = canonical
        search = None
        if valid and canonical["type"] == "branch" and args.branch_search_depth > 0:
            decision = choose_branch(
                state, canonical["cell"], canonical["value"],
                depth=args.branch_search_depth,
                node_budget=args.branch_search_budget)
            search = {"model_value": canonical["value"],
                      "chosen_value": decision["value"],
                      "override": decision["override"],
                      "survivors": decision["search"]["survivors"] if decision["search"] else None,
                      "feasible": decision["search"]["feasible"] if decision["search"] else None,
                      "depth": args.branch_search_depth}
            if decision["override"]:
                candidate = {"type": "branch", "cell": canonical["cell"],
                             "value": decision["value"]}
                valid, reason, canonical = validate_action(board, clue, candidate, stack, pending)
                if not valid:  # should not happen; keep the model choice
                    canonical = model_action
                    search["override"] = False

        event = {"step": step_index, "state": state, "raw": raw,
                 "action": canonical if valid else action,
                 "model_action": model_action,
                 "branch_search": search,
                 "legal_action_count": len(actions), "valid": valid, "reason": reason}
        if args.compact:
            event.pop("state", None)
        history.append(event)
        if not valid:
            failure = reason
            break
        kind = canonical["type"]
        if kind == "deduce":
            cell = canonical["cell"]
            r, c = cell["row"] - 1, cell["column"] - 1
            board[r][c] = sum(BIT[x] for x in canonical["candidates"])
        elif kind == "branch":
            cell = canonical["cell"]
            r, c = cell["row"] - 1, cell["column"] - 1
            stack.append({"to_step": step_index, "cell": cell,
                          "value": canonical["value"], "checkpoint": snapshot(board)})
            board[r][c] = BIT[canonical["value"]]
        elif kind == "contradiction":
            if not stack:
                failure = "contradiction_without_branch"
                break
            pending = True
        elif kind == "undo":
            branch = stack.pop()
            board = snapshot(branch["checkpoint"])
            r, c = branch["cell"]["row"] - 1, branch["cell"]["column"] - 1
            board[r][c] &= ~BIT[branch["value"]]
            pending = False
        elif kind == "finish":
            break
    else:
        failure = "action_limit"

    grid = [[one(mask) for mask in row] for row in board]
    reference = [[tile or "XX" for tile in row] for row in record["solution"]]
    correct = sum(a == b for row_a, row_b in zip(grid, reference)
                  for a, b in zip(row_a, row_b))
    finished = bool(history and history[-1]["valid"]
                    and history[-1]["action"]["type"] == "finish")
    return {
        "puzzle_id": record["puzzle_id"],
        "solved": finished and grid == reference,
        "valid_complete_board": finished and completed_error(board, clue) is None,
        "finished": finished, "failure": failure,
        "correct_cells": correct,
        "total_cells": len(reference) * len(reference[0]),
        "actions": len(history),
        "search_overrides": sum(1 for event in history
                                if event.get("branch_search")
                                and event["branch_search"]["override"]),
        "seconds": round(time.monotonic() - started, 3),
        "prediction": grid, "reference": reference, "history": history,
    }


def filter_only(puzzles, spec):
    """Restrict a puzzle list to the ids in ``spec`` (comma separated)."""
    wanted = [part.strip() for part in spec.split(",") if part.strip()]
    by_id = {record["puzzle_id"]: record for record in puzzles}
    missing = [pid for pid in wanted if pid not in by_id]
    if missing:
        raise SystemExit(f"--only: not in this split/sample: {missing}")
    return [by_id[pid] for pid in wanted]


def load_policy(args):
    """Load the adapter and return ``policy(state, actions) -> raw text``.

    Shared by the evaluator and the DAgger collector so both see the identical
    tokenizer, dtype and chat template.
    """
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            args.adapter, local_files_only=args.local_files_only)
    except (OSError, ValueError):
        # Older adapters may only contain weights/config; the base tokenizer is
        # the same one training used.
        tokenizer = AutoTokenizer.from_pretrained(
            args.model, local_files_only=args.local_files_only)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # bf16 requires sm_80+.  `is_bf16_supported()` can return True on some T4
    # images, where bf16 then falls back to slow emulation.
    capability = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
    bf16_ok = capability[0] >= 8 and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16_ok else torch.float16
    base = AutoModelForCausalLM.from_pretrained(
        args.model, dtype=dtype, attn_implementation="sdpa",
        local_files_only=args.local_files_only)
    model = PeftModel.from_pretrained(base, str(args.adapter)).to("cuda").eval()
    model.config.use_cache = True
    return model_policy(model, tokenizer, args)


def main():
    args = parse_args()

    if args.dataset == "medium":
        source = [json.loads(line) for line in args.data.open(encoding="utf-8")]
        puzzles = [record for record in source if record.get("split") == "test_medium"]
        if not puzzles:
            raise SystemExit("No test_medium puzzles in the data file")
        split = "test_medium"
        sample_strategy = "full_set"
        if args.only:
            puzzles = filter_only(puzzles, args.only)
            sample_strategy = "explicit_ids"
        elif args.limit:
            puzzles = select_medium_sample(puzzles, args.limit)
            sample_strategy = "deterministic_board_size_stratified"
    else:
        source = [json.loads(line) for line in args.small_data.open(encoding="utf-8")]
        dev_ids = development_ids(record["puzzle_id"] for record in source)
        puzzles = [record for record in source if record["puzzle_id"] in dev_ids]
        puzzles.sort(key=lambda record: record["puzzle_id"])
        split = "internal_puzzle_holdout_from_212_training_puzzles"
        sample_strategy = "dev_holdout"
        if args.only:
            puzzles = filter_only(puzzles, args.only)
            sample_strategy = "explicit_ids"
        elif args.limit:
            puzzles = puzzles[:args.limit]

    results = []
    policy = load_policy(args)
    for index, record in enumerate(puzzles, 1):
        result = rollout(policy, record, args)
        results.append(result)
        print(f"{args.dataset} evaluation: {index}/{len(puzzles)} "
              f"({result['puzzle_id']}: {'solved' if result['solved'] else result['failure']})",
              flush=True)

    report = {
        "run_id": args.adapter.parent.name if args.adapter.name == "adapter" else args.adapter.name,
        "dataset": args.dataset,
        "split": split,
        "sample_strategy": sample_strategy,
        "branch_search_depth": args.branch_search_depth,
        "attempted": len(results),
        "solved": sum(r["solved"] for r in results),
        "valid_complete_boards": sum(r["valid_complete_board"] for r in results),
        "correct_cells": sum(r["correct_cells"] for r in results),
        "total_cells": sum(r["total_cells"] for r in results),
        "failures": {reason: sum(r["failure"] == reason for r in results)
                     for reason in sorted({r["failure"] for r in results if r["failure"]})},
        "search_overrides": sum(r["search_overrides"] for r in results),
        "puzzles": results,
    }
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    summary = {key: value for key, value in report.items() if key != "puzzles"}
    summary["output"] = str(args.output)
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()


## 5. Checks

`tt_dagger.py --help` is included because its collection mode loads the adapter and
its `--build` mode is pure CPU; a broken import in either is cheaper to catch here
than 70 minutes into the rollout.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(parents=True, exist_ok=True)


def run(cmd, name, extra_env=None):
    """Run a command, stream and log its output, raise on non-zero exit."""
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd), flush=True)
    log_path = LOGS / f"{name}.log"
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}
    if extra_env:
        env.update(extra_env)
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, env=env, bufsize=1)
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
    if proc.wait():
        raise RuntimeError(f"{name} failed; full output in {log_path}")
    print(f"[{name} ok] log: {log_path}", flush=True)


import torch

print("python     :", sys.executable)
print("accelerate :", shutil.which("accelerate") or "NOT FOUND")
print("torch      :", torch.__version__, "| cuda", torch.version.cuda,
      "| available", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())
assert torch.cuda.is_available(), "torch cannot see a GPU; check the accelerator setting and the pip step"
assert torch.cuda.device_count() >= 1, "no CUDA devices"

LAUNCH = [sys.executable, "-m", "accelerate.commands.launch",
          "--multi_gpu", "--num_processes", "2",
          "--mixed_precision", "fp16", "--main_process_port", "29501"]
print("launcher   :", " ".join(LAUNCH[:3]), "..." if torch.cuda.device_count() >= 2
      else "(only one GPU: DDP steps will be skipped)")


In [ ]:
%cd /kaggle/working/tt
run([sys.executable, "-m", "py_compile", "tt_core.py", "tt_lib.py", "tt_prep.py",
     "tt_dagger.py", "tt_train.py", "tt_eval.py"], "compile")
run([sys.executable, "-c", "import importlib.util; found = importlib.util.find_spec('torchao') is not None; print('torchao installed:', found); raise SystemExit(1 if found else 0)"], "torchao-check")
run([sys.executable, "-c", "import tt_core, tt_lib, tt_prep, tt_dagger, tt_train, tt_eval; print('module imports ok')"], "imports")
run([sys.executable, "tt_dagger.py", "--help"], "dagger-help")
run([sys.executable, "tt_train.py", "--help"], "train-help")
run([sys.executable, "tt_eval.py", "--help"], "eval-help")


### Cache the base weights once

Both shards need the same ~3.1 GB checkpoint; fetching it twice at once makes them
fight over the HF cache lock. This also warms the cache for the training and eval
cells below.


In [ ]:
%cd /kaggle/working/tt
run([sys.executable, "-c",
     "from huggingface_hub import snapshot_download; "
     f"snapshot_download('{MODEL}'); print('cached {MODEL}')"], "model-cache")


## 6. Pick the rollout puzzles

The split is the same one run-05 used: 32 eval ids (asserted equal to the ones
run-04 measured), 32 fresh, 240 train. DAgger rollouts are drawn from the **train**
pool (32 puzzles) and the **fresh** pool (8 puzzles, used as dev loss). The eval ids
are asserted absent from both, so the comparison stays like-for-like.

Selection is by puzzle-id hash, so the same 40 puzzles are chosen on every machine.


In [ ]:
%cd /kaggle/working/tt
import hashlib
import json
import tt_prep

records = [json.loads(line) for line in
           (DATA / "train-puzzles-medium.jsonl").open(encoding="utf-8")]
medium = [record for record in records if record.get("split") == "test_medium"]
train_records, eval_records, fresh_records = tt_prep.split_pool(medium)
EVAL_IDS = {record["puzzle_id"] for record in eval_records}
print(f"pool: train {len(train_records)} | eval {len(eval_records)} | "
      f"fresh {len(fresh_records)}")


def pick(pool, count, salt):
    ordered = sorted(pool, key=lambda record: hashlib.sha256(
        (salt + record["puzzle_id"]).encode()).hexdigest())
    return [record["puzzle_id"] for record in ordered[:count]]


ROLL_TRAIN = pick(train_records, 32, "dagger-train-42")
ROLL_DEV = pick(fresh_records, 8, "dagger-dev-42")
assert not (set(ROLL_TRAIN) | set(ROLL_DEV)) & EVAL_IDS, "eval ids leaked into rollouts"
assert not set(ROLL_TRAIN) & set(ROLL_DEV)
print(f"rollout train {len(ROLL_TRAIN)}: {ROLL_TRAIN}")
print(f"rollout dev   {len(ROLL_DEV)}: {ROLL_DEV}")

DAGGER = Path("/kaggle/working/dagger")
DAGGER.mkdir(parents=True, exist_ok=True)
ALL_ROLL = ROLL_TRAIN + ROLL_DEV


## 7. Collect the policy's own branch states

Each shard loads its own copy of the adapter and rolls out 20 puzzles: depth 2, 256
actions, 16 new tokens — the same policy that produces the failures. Two shards run
concurrently, one per GPU, so wall time is ~70-85 min rather than twice that.

For every branch decision the collector stores the exact board (byte-identical to
what the evaluator renders), the audited value, whether that value was still in the
menu, and what the policy committed. A shard whose JSONL already has content is
skipped, so a timeout only redoes the missing half.


In [ ]:
%cd /kaggle/working/tt
from concurrent.futures import ThreadPoolExecutor

SHARDS = [ALL_ROLL[index::2] for index in range(2)]


def collect_shard(index):
    ids = SHARDS[index]
    output = DAGGER / f"dagger-shard{index}.jsonl"
    if output.exists() and output.stat().st_size > 0:
        print(f"[shard {index}] {output.name} already has content, skipping")
        return index, 0
    device = str(index)
    cmd = [sys.executable, "tt_dagger.py",
           "--adapter", str(ADAPTER),
           "--data", str(DATA / "train-puzzles-medium.jsonl"),
           "--model", MODEL,
           "--only", ",".join(ids),
           "--max-actions", "256",
           "--branch-search-depth", "2",
           "--max-new-tokens", "16",
           "--output", str(output)]
    log_path = LOGS / f"dagger-shard{index}.log"
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false",
           "CUDA_VISIBLE_DEVICES": device}
    print(f"[shard {index}] GPU {device}, {len(ids)} puzzles -> {log_path}", flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                                text=True, env=env)
        status = proc.wait()
    print(f"[shard {index}] exit {status}", flush=True)
    return index, status


parallel = torch.cuda.device_count() >= len(SHARDS)
if parallel:
    with ThreadPoolExecutor(max_workers=len(SHARDS)) as pool:
        outcomes = list(pool.map(collect_shard, range(len(SHARDS))))
else:
    print("one GPU visible: shards run one after the other")
    outcomes = [collect_shard(index) for index in range(len(SHARDS))]

failed = [index for index, status in outcomes if status]
for index in failed:
    tail = (LOGS / f"dagger-shard{index}.log").read_text(encoding="utf-8").splitlines()[-25:]
    print(f"\n--- shard {index} tail ---")
    print("\n".join(tail))
if len(failed) == len(SHARDS):
    raise RuntimeError("both shards failed; see /kaggle/working/logs/dagger-shard*.log")
print("shards failed:", failed)


In [ ]:
import json

DAGGER = Path("/kaggle/working/dagger")
states = []
for path in sorted(DAGGER.glob("dagger-shard*.jsonl")):
    chunk = [json.loads(line) for line in path.open(encoding="utf-8")]
    print(f"{path.name}: {len(chunk)} states")
    states.extend(chunk)

got_ids = {item["puzzle_id"] for item in states}
missing = [pid for pid in ALL_ROLL if pid not in got_ids]
correct = sum(item["model"] == item["truth"] for item in states)
unrecoverable = sum(not item["recoverable"] for item in states)
print(f"\npuzzles {len(got_ids)}/{len(ALL_ROLL)} | states {len(states)} | "
      f"model correct {correct} ({correct / max(1, len(states)):.0%})")
print(f"unrecoverable (truth no longer offered): {unrecoverable}")
print(f"missing: {missing}")
assert len(got_ids & EVAL_IDS) == 0, "an eval puzzle was rolled out"


## 8. Build the branch-only training rows

`--build` labels every recoverable state with the audited value, emits both menu
orders, and drops repeated boards (a revisit after undo is the same prompt). States
whose true tile was eliminated are counted as `unrecoverable` and skipped: training
the other value there would teach the model to commit to a poisoned board.

**Measured on the first collection** (40 puzzles asked, 39 delivered):

| | Train | Dev |
| --- | --- | --- |
| Branch states | 334 | 51 |
| Model already correct on its own board | 244 (73%) | 38 (75%) |
| Unrecoverable (truth no longer offered) | 51 (15%) | 10 (20%) |
| Labelled rows | **283** | 41 |
| Training rows after both menu orders | **566** | 41 |

So the useful signal is **42 teachable wrong decisions**, 23 of them the eval failure
signature (model says `XX` where a tile belongs). One dev puzzle never reached a
branch in 256 actions, which is why dev is 7 puzzles: that is a warning below, not a
failure.


In [ ]:
%cd /kaggle/working/tt
STATES = ",".join(str(path) for path in sorted(DAGGER.glob("dagger-shard*.jsonl")))
for role, ids in (("train", ROLL_TRAIN), ("dev", ROLL_DEV)):
    run([sys.executable, "tt_dagger.py", "--build", "--states", STATES,
         "--output-dir", str(DAGGER), "--role", role, "--only", ",".join(ids)],
        f"dagger-build-{role}")

train_report = json.loads((DAGGER / "train-report.json").read_text(encoding="utf-8"))
dev_report = json.loads((DAGGER / "dev-report.json").read_text(encoding="utf-8"))
print(json.dumps({role: {key: report[key] for key in
                         ("states", "unique_states", "model_correct", "model_wrong",
                          "unrecoverable", "labelled_rows", "examples",
                          "estimated_tokens")}
                  for role, report in (("train", train_report), ("dev", dev_report))},
                 indent=1))

STATE_IDS = {item["puzzle_id"] for item in states}
missing_train = sorted(set(ROLL_TRAIN) - STATE_IDS)
missing_dev = sorted(set(ROLL_DEV) - STATE_IDS)
print(f"\npuzzles with no branch states: train {len(missing_train)}, dev {len(missing_dev)}")
print("  train:", missing_train)
print("  dev  :", missing_dev)

# A puzzle that never stalls never offers a branch, so it produces no state. That is
# a smaller pool, not a data error: warn and continue, otherwise one quiet puzzle
# aborts the run after the collection has already been paid for.
assert train_report["examples"] == train_report["labelled_rows"] * 2, "order doubling broke"
assert train_report["labelled_rows"] >= 40, (
    f"only {train_report['labelled_rows']} labelled rows: the collector found almost "
    f"no usable branch states, check logs/dagger-shard*.log before training")
if missing_train:
    print("WARNING: training puzzles with no branch states:", missing_train)
if missing_dev:
    print(f"WARNING: dev covers {dev_report['puzzles']} puzzles, not {len(ROLL_DEV)}:",
          missing_dev)


### One row, and the label it carries

Every row is a branch decision with the audited value as the target — including the
states where the policy guessed the other value. The menu sits at the end of the
prompt, which is why `max_length` stays at 4096.


In [ ]:
first = None
with (DAGGER / "train-examples.jsonl").open(encoding="utf-8") as handle:
    for line in handle:
        first = json.loads(line)
        break

payload = json.loads(first["user"].splitlines()[1])
menu = {entry["id"]: entry["action"] for entry in payload["legal_actions"]}
chosen = json.loads(first["completion"])["choice"]
print("puzzle    :", first["puzzle_id"], "| kind:", first["kind"])
print("completion:", first["completion"], "->", menu[chosen])
print("prompt    : %d chars, %d menu entries" % (len(first["user"]), len(menu)))


## 9. Smoke train

16 rows, 4 optimizer steps, then the same under DDP. ~6 min.


In [ ]:
%cd /kaggle/working/tt

smoke = [sys.executable, "tt_train.py",
         "--examples", str(DAGGER / "train-examples.jsonl"),
         "--dev-examples", str(DAGGER / "dev-examples.jsonl"),
         "--init-adapter", str(ADAPTER),
         "--lr", "2e-4",
         "--max-length", "4096",
         "--smoke"]

run(smoke + ["--output", "/kaggle/working/smoke-dagger"], "smoke-dagger")
if torch.cuda.device_count() >= 2:
    run(LAUNCH + smoke[1:] + ["--output", "/kaggle/working/smoke-dagger-ddp"],
        "smoke-dagger-ddp")
else:
    print("only one GPU: skipping the DDP smoke test")


## 10. Train on the branch rows only

No deduction rows at all, so every gradient step is on a decision this run is trying
to change. With the measured set (566 rows) three epochs is **~53 optimizer steps** at
2e-4 — twice run-05's learning rate, because run-05's branch logits barely moved — and
the dev set is now the same distribution as training (the policy's own boards on 7
unseen puzzles), so its loss is informative rather than cosmetic.

Watch two things in the report:

* `development_loss_after` below `before` — it learned the own-state distribution;
* the probe afterwards — whether that changed any decision. Run-05 proved those two
  can disagree, which is why section 11 reports branch sequences too.

**If you only want the adapter** (stopping at run-04's 28/32), this is the last
expensive cell: run 14 straight after it and skip 11–13. Section 14 zips
`run-dagger/adapter` for download.


In [ ]:
%cd /kaggle/working/tt

train_cmd = [sys.executable, "tt_train.py",
             "--examples", str(DAGGER / "train-examples.jsonl"),
             "--dev-examples", str(DAGGER / "dev-examples.jsonl"),
             "--init-adapter", str(ADAPTER),
             "--output", "/kaggle/working/run-dagger",
             "--epochs", "3",
             "--lr", "2e-4",
             "--max-length", "4096",
             "--batch-size", "1",
             "--grad-accum", "16"]

if torch.cuda.device_count() >= 2:
    try:
        run(LAUNCH + train_cmd[1:], "train-dagger-ddp")
    except (RuntimeError, FileNotFoundError) as error:
        print("\nDDP failed:", error)
        print("Falling back to a single GPU.\n")
        run(train_cmd, "train-dagger-single")
else:
    run(train_cmd, "train-dagger-single")


In [ ]:
import json

NEW_ADAPTER = Path("/kaggle/working/run-dagger/adapter")
print("adapter:", NEW_ADAPTER, "| files:", len(list(NEW_ADAPTER.glob('*'))))
report = json.loads((Path("/kaggle/working/run-dagger") / "training.json").read_text(encoding="utf-8"))
print(json.dumps({key: report[key] for key in
                  ("puzzles_train", "puzzles_dev", "train_examples", "dev_examples",
                   "optimizer_steps", "development_loss_before", "development_loss_after",
                   "train_runtime_seconds", "peak_gpu_memory_gib")}, indent=1))


## 11. Probe: the same 10 puzzles

Identical settings to run-05's probe (depth 2, 512 actions, 16 tokens), so the only
difference is the adapter. 5/5 across two GPUs, ~40 min.

The second cell is the one that matters: it compares the branch **sequence** with
run-04's, embedded below. Run-05 changed 3 sequences out of 10 and gained nothing; a
policy that has actually learned the hard states should differ on most of them.


In [ ]:
%cd /kaggle/working/tt
from concurrent.futures import ThreadPoolExecutor

PROBE_IDS = [
    "03092026-medium",
    "10012026-medium",
    "13072026-medium",
    "14072026-medium",
    "18102026-medium",
    "19042026-medium",
    "22012026-medium",
    "24032026-medium",
    "25022026-medium",
    "30082026-medium"
]


def eval_ids(ids, tag, shard_count=2, skip_existing=True):
    """Evaluate ids on shard_count GPUs (or serially), one JSON per shard."""
    ids = list(dict.fromkeys(ids))
    if not ids:
        print(f"[{tag}] nothing left to evaluate")
        return []
    adapter_mtime = max(path.stat().st_mtime for path in NEW_ADAPTER.glob("*"))
    shards = [ids[index::shard_count] for index in range(shard_count)]
    shards = [shard for shard in shards if shard]
    parallel = torch.cuda.device_count() >= len(shards)
    devices = ([str(index) for index in range(len(shards))] if parallel
               else ["0"] * len(shards))

    def one(index):
        shard, device = shards[index], devices[index]
        output = Path(f"/kaggle/working/{tag}-shard{index}.json")
        if skip_existing and output.exists():
            done = json.loads(output.read_text(encoding="utf-8"))["attempted"]
            stale = output.stat().st_mtime < adapter_mtime
            if done >= len(shard) and not stale:
                print(f"[{tag} {index}] {output.name} already complete, skipping")
                return index, 0
            if stale:
                print(f"[{tag} {index}] {output.name} predates the adapter, re-running")
        cmd = [sys.executable, "tt_eval.py",
               "--adapter", str(NEW_ADAPTER),
               "--data", str(DATA / "train-puzzles-medium.jsonl"),
               "--dataset", "medium",
               "--model", MODEL,
               "--branch-search-depth", "2",
               "--max-actions", "512",
               "--max-new-tokens", "16",
               "--only", ",".join(shard),
               "--output", str(output),
               "--compact"]
        log_path = LOGS / f"{tag}-shard{index}.log"
        env = {**os.environ, "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false",
               "CUDA_VISIBLE_DEVICES": device}
        print(f"[{tag} {index}] GPU {device}, {len(shard)} puzzles -> {log_path}", flush=True)
        with log_path.open("w", encoding="utf-8") as log:
            proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                                    text=True, env=env)
            status = proc.wait()
        print(f"[{tag} {index}] exit {status}", flush=True)
        return index, status

    if parallel:
        with ThreadPoolExecutor(max_workers=len(shards)) as pool:
            outcomes = list(pool.map(one, range(len(shards))))
    else:
        print(f"[{tag}] fewer than {len(shards)} GPUs visible: shards run serially")
        outcomes = [one(index) for index in range(len(shards))]

    failed = [index for index, status in outcomes if status]
    for index in failed:
        tail = (LOGS / f"{tag}-shard{index}.log").read_text(encoding="utf-8").splitlines()[-25:]
        print(f"\n--- {tag} shard {index} tail ---")
        print("\n".join(tail))
    if len(failed) == len(shards):
        raise RuntimeError(f"all {tag} shards failed; see /kaggle/working/logs/{tag}-shard*.log")
    return failed


print("probe shards failed:", eval_ids(PROBE_IDS, "probe"))


In [ ]:
import json

RUN04_SEQUENCES = {"03092026-medium": [[1, 2, "XX"], [1, 3, "XX"], [1, 4, "XX"], [1, 5, "XX"], [1, 6, "XX"], [1, 7, "XX"], [2, 2, "SW"], [2, 3, "XX"], [2, 4, "XX"], [2, 6, "EW"], [2, 7, "EW"], [2, 5, "EW"], [2, 2, "SW"], [2, 3, "XX"], [2, 4, "XX"], [4, 2, "XX"], [4, 3, "XX"], [4, 4, "XX"], [2, 5, "EW"]], "10012026-medium": [[1, 1, "XX"], [1, 2, "XX"], [1, 3, "XX"], [1, 4, "XX"], [1, 5, "SE"], [2, 1, "XX"], [2, 2, "XX"], [2, 3, "XX"], [2, 4, "XX"], [3, 1, "XX"], [3, 2, "XX"], [3, 3, "XX"], [4, 1, "XX"], [3, 4, "NS"]], "13072026-medium": [[3, 1, "XX"], [3, 2, "NS"], [3, 3, "XX"], [3, 4, "XX"], [3, 5, "XX"], [3, 7, "EW"], [4, 1, "XX"], [4, 2, "NS"], [4, 3, "XX"], [4, 3, "EW"], [4, 7, "XX"], [4, 3, "XX"], [4, 1, "SE"]], "14072026-medium": [[1, 2, "XX"], [1, 3, "XX"], [1, 4, "XX"], [1, 5, "XX"], [1, 6, "XX"], [1, 7, "XX"], [1, 4, "EW"], [3, 3, "EW"], [3, 4, "EW"], [3, 6, "XX"], [3, 7, "XX"], [3, 8, "XX"], [3, 9, "XX"], [3, 5, "NS"], [3, 6, "XX"], [3, 7, "XX"], [3, 8, "XX"], [3, 9, "XX"], [4, 2, "XX"], [4, 3, "XX"], [4, 4, "NS"], [4, 5, "XX"], [4, 6, "NS"], [4, 7, "XX"], [4, 7, "EW"], [4, 8, "XX"]], "18102026-medium": [[1, 3, "EW"], [2, 2, "NS"], [2, 6, "EW"], [3, 5, "XX"], [3, 6, "XX"], [4, 2, "XX"], [2, 7, "XX"], [3, 4, "EW"], [3, 5, "EW"], [4, 2, "NE"]], "19042026-medium": [[1, 2, "XX"], [1, 3, "XX"], [1, 4, "SE"], [5, 6, "NS"], [6, 2, "NS"], [6, 3, "NE"], [7, 2, "NS"], [7, 3, "XX"], [7, 4, "XX"], [7, 2, "XX"], [7, 3, "XX"], [7, 4, "NS"], [7, 5, "EW"], [8, 2, "XX"], [6, 2, "NS"], [6, 3, "NE"], [7, 2, "NS"], [7, 3, "XX"], [7, 4, "XX"]], "22012026-medium": [[3, 1, "XX"], [3, 2, "XX"], [3, 3, "SE"], [3, 7, "XX"], [3, 8, "XX"], [3, 9, "XX"], [3, 7, "XX"], [3, 8, "XX"], [3, 3, "EW"], [4, 1, "XX"]], "24032026-medium": [[1, 1, "XX"], [1, 2, "XX"], [1, 3, "XX"], [1, 4, "XX"], [1, 5, "XX"], [1, 6, "XX"], [1, 7, "SE"], [2, 1, "XX"], [2, 2, "XX"], [2, 3, "XX"], [2, 1, "XX"], [2, 2, "XX"], [2, 3, "XX"], [2, 4, "NS"], [5, 2, "XX"], [5, 3, "XX"], [5, 4, "NS"], [7, 6, "EW"], [8, 1, "SE"]], "25022026-medium": [[1, 2, "XX"], [1, 3, "XX"], [1, 4, "XX"], [1, 5, "XX"], [1, 6, "EW"], [1, 7, "EW"], [1, 5, "EW"], [1, 6, "EW"], [1, 4, "EW"], [1, 5, "EW"], [2, 4, "XX"], [2, 6, "NE"], [3, 3, "EW"], [3, 6, "SE"], [4, 5, "NE"], [5, 2, "NS"], [5, 3, "XX"], [6, 2, "NE"], [6, 3, "EW"], [7, 2, "XX"], [7, 3, "XX"], [6, 4, "NS"], [7, 2, "XX"]], "30082026-medium": [[1, 1, "XX"], [1, 2, "XX"], [1, 3, "XX"], [2, 1, "XX"], [2, 3, "EW"], [2, 6, "NS"], [3, 3, "XX"], [3, 4, "XX"], [3, 3, "NS"], [4, 2, "EW"], [2, 6, "NE"], [3, 3, "XX"], [3, 4, "NE"], [4, 2, "XX"], [4, 3, "XX"], [3, 5, "XX"], [4, 2, "XX"], [4, 3, "NS"], [4, 4, "EW"], [6, 2, "XX"]]}

probe = [json.loads(path.read_text(encoding="utf-8"))
         for path in sorted(Path("/kaggle/working").glob("probe-shard*.json"))]
got = {p["puzzle_id"]: p for report in probe for p in report["puzzles"]}
print(f"probe results: {len(got)}/{len(PROBE_IDS)}")


def sequence(history):
    out = []
    for event in history:
        action = event.get("action") or {}
        if action.get("type") == "branch":
            cell = action["cell"]
            out.append([cell["row"], cell["column"], action["value"]])
    return out


solved_now = [pid for pid in PROBE_IDS if pid in got and got[pid]["solved"]]
identical = [pid for pid in PROBE_IDS if pid in got
             and sequence(got[pid]["history"]) == RUN04_SEQUENCES[pid]]
print(f"solved at 512: {len(solved_now)}/{len(got)} (run-04: 6/10, run-05: 6/10)")
print("solved :", solved_now)
print("failed :", [pid for pid in PROBE_IDS if pid in got and not got[pid]["solved"]])
print(f"branch sequences identical to run-04: {len(identical)}/{len(got)} {identical}")
print("=> policy changed" if len(identical) < len(got) else
      "=> policy did NOT change: the states were still not the binding ones")


## 12. Remaining 22 ids

Only run this if the probe improved on 6/10. With the same four targets still at the
512-action cap, the score cannot exceed 28/32, and this run's ceiling
matters: the point is 29+.

If the probe is still 6/10, skip to section 13 with only the probe loaded — the
comparison table will print `PARTIAL`, which is the honest label.


In [ ]:
%cd /kaggle/working/tt

ALL_IDS = [
    "01012026-medium",
    "02022026-medium",
    "03032026-medium",
    "03092026-medium",
    "04062026-medium",
    "05022026-medium",
    "05082026-medium",
    "06052026-medium",
    "08102026-medium",
    "10012026-medium",
    "12062026-medium",
    "13072026-medium",
    "14072026-medium",
    "17042026-medium",
    "18102026-medium",
    "19042026-medium",
    "19062026-medium",
    "19102026-medium",
    "20012026-medium",
    "20062026-medium",
    "22012026-medium",
    "22022026-medium",
    "24032026-medium",
    "24102026-medium",
    "25022026-medium",
    "27012026-medium",
    "28052026-medium",
    "28102026-medium",
    "29052026-medium",
    "29082026-medium",
    "30032026-medium",
    "30082026-medium"
]

done = set(got)
remaining = [pid for pid in ALL_IDS if pid not in done]
print(f"{len(done)} covered by the probe, {len(remaining)} to go")
print("full shards failed:", eval_ids(remaining, "full"))


## 13. Compare

`run04@512` is whether that puzzle solved for run-04 at 512 actions (28/32).
Gains and losses are printed separately: the target ids are
`03092026, 14072026, 25022026, 30082026`, and losing a solved puzzle cancels a gain.
`invalid` must stay 0.


In [ ]:
import json
import pprint
from pathlib import Path
import pandas as pd

ALL_IDS = [
    "01012026-medium",
    "02022026-medium",
    "03032026-medium",
    "03092026-medium",
    "04062026-medium",
    "05022026-medium",
    "05082026-medium",
    "06052026-medium",
    "08102026-medium",
    "10012026-medium",
    "12062026-medium",
    "13072026-medium",
    "14072026-medium",
    "17042026-medium",
    "18102026-medium",
    "19042026-medium",
    "19062026-medium",
    "19102026-medium",
    "20012026-medium",
    "20062026-medium",
    "22012026-medium",
    "22022026-medium",
    "24032026-medium",
    "24102026-medium",
    "25022026-medium",
    "27012026-medium",
    "28052026-medium",
    "28102026-medium",
    "29052026-medium",
    "29082026-medium",
    "30032026-medium",
    "30082026-medium"
]
BASELINE = {'01012026-medium': [100, 100, True],
 '02022026-medium': [90, 90, True],
 '03032026-medium': [100, 100, True],
 '03092026-medium': [23, 81, False],
 '04062026-medium': [90, 90, True],
 '05022026-medium': [100, 100, True],
 '05082026-medium': [90, 90, True],
 '06052026-medium': [90, 90, True],
 '08102026-medium': [81, 81, True],
 '10012026-medium': [76, 90, True],
 '12062026-medium': [90, 90, True],
 '13072026-medium': [40, 90, True],
 '14072026-medium': [35, 100, False],
 '17042026-medium': [81, 81, True],
 '18102026-medium': [71, 81, True],
 '19042026-medium': [64, 90, True],
 '19062026-medium': [90, 90, True],
 '19102026-medium': [100, 100, True],
 '20012026-medium': [81, 81, True],
 '20062026-medium': [90, 90, True],
 '22012026-medium': [76, 100, True],
 '22022026-medium': [90, 90, True],
 '24032026-medium': [20, 90, True],
 '24102026-medium': [90, 90, True],
 '25022026-medium': [40, 100, False],
 '27012026-medium': [81, 81, True],
 '28052026-medium': [100, 100, True],
 '28102026-medium': [90, 90, True],
 '29052026-medium': [81, 81, True],
 '29082026-medium': [90, 90, True],
 '30032026-medium': [81, 81, True],
 '30082026-medium': [49, 90, False]}
TARGET = [
    "03092026-medium",
    "14072026-medium",
    "25022026-medium",
    "30082026-medium"
]

reports = []
adapter_mtime = max(path.stat().st_mtime for path in NEW_ADAPTER.glob("*"))
for path in sorted(Path("/kaggle/working").glob("*-shard*.json")):
    if path.stat().st_mtime < adapter_mtime:
        print("skipping", path.name, "(older than the adapter)")
        continue
    reports.append(json.loads(path.read_text(encoding="utf-8")))
    print("loaded", path.name, "|", reports[-1]["attempted"], "puzzles")

got = {p["puzzle_id"]: p for report in reports for p in report["puzzles"]}
missing = [pid for pid in ALL_IDS if pid not in got]
print("covered:", len(got), "/", len(ALL_IDS), "| missing:", missing)

rows = []
for pid in ALL_IDS:
    if pid not in got:
        continue
    new = got[pid]
    correct_256, total_256, solved_512 = BASELINE[pid]
    invalid = sum(1 for event in new.get("history") or [] if not event.get("valid"))
    rows.append({
        "puzzle": pid.replace("-medium", ""),
        "target": pid in TARGET,
        "cells@256": f"{correct_256}/{total_256}",
        "run04@512": solved_512,
        "cells@new": f"{new['correct_cells']}/{new['total_cells']}",
        "new": new["solved"],
        "actions": new["actions"],
        "overrides": new["search_overrides"],
        "invalid": invalid,
        "min": round(new["seconds"] / 60, 1),
    })

frame = pd.DataFrame(rows).sort_values(["target", "puzzle"], ascending=[False, True])
display(frame)

new_solved = {row["puzzle"] + "-medium" for row in rows if row["new"]}
base_solved = {pid for pid in ALL_IDS if BASELINE[pid][2]}
gained = sorted(new_solved - base_solved)
lost = sorted(base_solved - new_solved)
invalid_total = sum(row["invalid"] for row in rows)

print()
print(f"score: {len(base_solved)}/32 (run-04 @512) -> {len(new_solved)}/32 (new adapter)")
print(f"gained: {[pid.replace('-medium', '') for pid in gained]}")
print(f"lost  : {[pid.replace('-medium', '') for pid in lost]}")
print(f"targets still failing: "
      f"{[pid.replace('-medium', '') for pid in sorted(set(TARGET) & (set(ALL_IDS) - new_solved))]}")
print(f"invalid actions: {invalid_total}")
if missing:
    print("PARTIAL: re-run section 12, then this cell")
print(f"eval time (sum over puzzles): {sum(p['seconds'] for p in got.values()) / 60:.0f} min")

merged = {
    "run_id": "run-06-medium-dagger",
    "branch_search_depth": 2,
    "max_actions": 512,
    "max_new_tokens": 16,
    "attempted": len(got),
    "solved": len(new_solved),
    "baseline_solved": len(base_solved),
    "gained": gained,
    "lost": lost,
    "invalid_actions": invalid_total,
    "missing": missing,
    "puzzles": [got[pid] for pid in ALL_IDS if pid in got],
}
Path("/kaggle/working/dagger-eval-merged.json").write_text(
    json.dumps(merged, indent=1), encoding="utf-8")
print("wrote /kaggle/working/dagger-eval-merged.json")


## 14. Save the artifacts

`dagger-eval-merged.json` holds the per-action histories; the state JSONLs and the
two build reports are what a next round would extend.


In [ ]:
import pathlib, shutil

art = pathlib.Path("/kaggle/working/artifacts")
art.mkdir(exist_ok=True)
for pattern in ("*-shard*.json", "dagger-eval-merged.json", "dagger/*.json",
                "dagger/*.jsonl", "run-dagger/training.json"):
    for path in pathlib.Path("/kaggle/working").glob(pattern):
        shutil.copy(path, art / path.name)

adapter_dir = pathlib.Path("/kaggle/working/run-dagger/adapter")
if adapter_dir.exists() and any(adapter_dir.glob("*")):
    zip_path = shutil.make_archive(str(art / "run-dagger-adapter"), "zip",
                                   str(adapter_dir))
    print("adapter zip:", zip_path)
else:
    zip_path = None
    print("NO ADAPTER: /kaggle/working/run-dagger/adapter is missing or empty.")
    print("That means section 10 did not finish; re-run it, then this cell.")
if LOGS.exists():
    shutil.copytree(LOGS, art / "logs", dirs_exist_ok=True)
print(sorted(p.name for p in art.iterdir()))
print("download the contents of", art)
